In [ ]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# ==========================================
# 1. SETUP & PATHS (MULTI-PLATE)
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2","PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2","PLATE5_T0","PLATE5_T1","PLATE5_T2"]

# --- ADDED: Define where you want to save the CSV ---
OUTPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")

TREATMENT_COL = "Treatment"
NUM_CHANNELS = 5
FEATS_PER_CH = 1280

all_plates_data = []

for plate_id in PLATES:
    print(f"\n--- Processing {plate_id} ---")
    
    FEATURES_BASE = os.path.join(PROJECT_ROOT,"features", plate_id)
    METADATA_PATH = os.path.join(PROJECT_ROOT,"metadata", f"index_{plate_id}.csv")
    
    if not os.path.exists(METADATA_PATH):
        print(f"Skipping {plate_id}: Metadata not found.")
        continue

    meta = pd.read_csv(METADATA_PATH)
    well_storage = {}
    well_to_treatment = {}

    for i in tqdm(meta.index, desc=f"Loading {plate_id}"):
        well_id = f"{plate_id}_{meta.loc[i, 'Metadata_Well']}"
        treatment = str(meta.loc[i, TREATMENT_COL]).strip()
        
        filename = os.path.join(FEATURES_BASE, 
                                str(meta.loc[i, "Metadata_Well"]), 
                                f"{meta.loc[i, 'Metadata_Site']}.npz")
        
        if os.path.isfile(filename):
            try:
                with np.load(filename) as data:
                    cells = data["features"]
                    cells_f = cells[~np.isnan(cells).any(axis=1)]
                    
                    if len(cells_f) > 0:
                        if well_id not in well_storage:
                            well_storage[well_id] = []
                            well_to_treatment[well_id] = treatment
                        well_storage[well_id].append(cells_f)
            except:
                continue

    for well_id, feature_list in well_storage.items():
        all_cells_in_well = np.vstack(feature_list)
        well_median = np.median(all_cells_in_well, axis=0)
        
        row = {"Plate": plate_id, "Well_ID": well_id, "Treatment": well_to_treatment[well_id]}
        for idx, val in enumerate(well_median):
            row[idx] = val
        all_plates_data.append(row)

# Combine everything into one giant DataFrame
wells_df = pd.DataFrame(all_plates_data)
print(f"\nAggregation complete. Total Wells: {len(wells_df)}")

# ==========================================
# 2. SAVE TO CSV
# ==========================================
# index=False prevents pandas from writing a column for the row numbers
wells_df.to_csv(OUTPUT_CSV, index=False)
print(f"Successfully saved to: {OUTPUT_CSV}")

In [ ]:
import umap
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text

# Define the channel subsets (same as before)
feature_cols = [i for i in range(NUM_CHANNELS * FEATS_PER_CH)]
plot_configs = [
    {"name": "All Channels", "indices": feature_cols},
    {"name": "Channel 1", "indices": list(range(0, 1280))},
    {"name": "Channel 2", "indices": list(range(1280, 2560))},
    {"name": "Channel 3", "indices": list(range(2560, 3840))},
    {"name": "Channel 4", "indices": list(range(3840, 5120))},
    {"name": "Channel 5", "indices": list(range(5120, 6400))}
]

# LOOP THROUGH EACH PLATE INDIVIDUALLY
for plate_id in wells_df['Plate'].unique():
    plate_subset = wells_df[wells_df['Plate'] == plate_id].copy()
    
    fig, axes = plt.subplots(2, 3, figsize=(26, 18))
    axes = axes.flatten()

    for idx, config in enumerate(plot_configs):
        ax = axes[idx]
        
        # 1. Prepare & Scale Data
        X_subset = plate_subset[config['indices']].values
        X_scaled = StandardScaler().fit_transform(X_subset)
        
        # 2. Run UMAP
        reducer = umap.UMAP(n_neighbors=min(15, len(plate_subset)-1), min_dist=0.1, random_state=42)
        embedding = reducer.fit_transform(X_scaled)
        
        # 3. Plotting logic (same as your original code)
        is_control = plate_subset[TREATMENT_COL] == "no_sgRNA"
        
        # Draw Control vs Mutants
        ax.scatter(embedding[is_control, 0], embedding[is_control, 1], 
                   c='red', marker='x', s=120, label='Control', zorder=4)
        ax.scatter(embedding[~is_control, 0], embedding[~is_control, 1], 
                   c='black', marker='o', s=50, alpha=0.5, label='Mutants', zorder=3)

        # 4. Text Labels (using subset data)
        texts = []
        for i, row in enumerate(plate_subset.itertuples()):
            color = 'red' if row.Treatment == "no_sgRNA" else 'black'
            label = row.Well_ID if row.Treatment == "no_sgRNA" else row.Treatment
            texts.append(ax.text(embedding[i, 0], embedding[i, 1], label, fontsize=7, color=color))

        adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='silver', lw=0.5))
        ax.set_title(f"{plate_id} - {config['name']}")

    plt.suptitle(f"UMAP Analysis: {plate_id}", fontsize=22, y=1.02)
    plt.tight_layout()
    # plt.savefig(f"umap_{plate_id}.png")
    plt.show()

In [ ]:
import umap
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text
import matplotlib.cm as cm
import numpy as np

# --- 1. Dynamic Color Setup ---
plate_list = sorted(wells_df['Plate'].unique())
num_plates = len(plate_list)

# Use a qualitative colormap (Set1, Set3, or Dark2 are good for distinct categories)
# 'tab10' or 'tab20' are excellent for up to 10 or 20 distinct categories
cmap = plt.get_cmap('tab10') 
plate_colors = [cmap(i) for i in np.linspace(0, 1, num_plates)]
plate_color_map = {plate: plate_colors[i] for i, plate in enumerate(plate_list)}

# --- 2. Configuration ---
feature_cols = [i for i in range(NUM_CHANNELS * FEATS_PER_CH)]
plot_configs = [
    {"name": "All Channels", "indices": feature_cols},
    {"name": "Channel 1", "indices": list(range(0, 1280))},
    {"name": "Channel 2", "indices": list(range(1280, 2560))},
    {"name": "Channel 3", "indices": list(range(2560, 3840))},
    {"name": "Channel 4", "indices": list(range(3840, 5120))},
    {"name": "Channel 5", "indices": list(range(5120, 6400))}
]

fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    print(f"Running UMAP for {config['name']}...")
    
    X_subset = wells_df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X_subset)
    
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    texts = []
    
    # 3. Plot Plate by Plate
    for plate in plate_list:
        mask = wells_df['Plate'] == plate
        plate_embed = embedding[mask]
        plate_meta = wells_df[mask]
        
        is_ctrl = plate_meta[TREATMENT_COL] == "no_sgRNA"
        
        # Plot Mutants (Filled Circles)
        ax.scatter(plate_embed[~is_ctrl, 0], plate_embed[~is_ctrl, 1], 
                   color=plate_color_map[plate], marker='o', s=60, alpha=0.6, 
                   label=f"{plate} (Mutant)")
        
        # Plot Controls (Open Squares)
        ax.scatter(plate_embed[is_ctrl, 0], plate_embed[is_ctrl, 1], 
                   edgecolors=plate_color_map[plate], facecolors='none', 
                   marker='s', s=130, linewidths=2, label=f"{plate} (Control)")

        # 4. Add labels
        for i, row in enumerate(plate_meta.itertuples()):
            label = f"{row.Treatment}_{row.Plate.split('_')[-1]}"
            texts.append(ax.text(plate_embed[i, 0], plate_embed[i, 1], label, 
                                 fontsize=6, color=plate_color_map[plate]))

    # Formatting
    ax.set_title(f"{config['name']}", fontsize=16, fontweight='bold')
    
    # Move legend to the side if it gets too crowded with 6+ plates
    if idx == 0:
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), markerscale=1, fontsize=8, ncol=1)
    
    # Adjust text to prevent overlapping labels
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.3))

plt.suptitle("Combined UMAP: Multi-Plate Analysis", fontsize=24, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# ==========================================
# 1. SETUP & PATHS (MULTI-PLATE)
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2","PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2","PLATE5_T0","PLATE5_T1","PLATE5_T2"]

# --- ADDED: Define where you want to save the CSV ---
OUTPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")

TREATMENT_COL = "Treatment"
NUM_CHANNELS = 5
FEATS_PER_CH = 1280

all_plates_data = []

for plate_id in PLATES:
    print(f"\n--- Processing {plate_id} ---")
    
    FEATURES_BASE = os.path.join(PROJECT_ROOT,"features", plate_id)
    METADATA_PATH = os.path.join(PROJECT_ROOT,"metadata", f"index_{plate_id}.csv")
    
    if not os.path.exists(METADATA_PATH):
        print(f"Skipping {plate_id}: Metadata not found.")
        continue

    meta = pd.read_csv(METADATA_PATH)
    well_storage = {}
    well_to_treatment = {}

    for i in tqdm(meta.index, desc=f"Loading {plate_id}"):
        well_id = f"{plate_id}_{meta.loc[i, 'Metadata_Well']}"
        treatment = str(meta.loc[i, TREATMENT_COL]).strip()
        
        filename = os.path.join(FEATURES_BASE, 
                                str(meta.loc[i, "Metadata_Well"]), 
                                f"{meta.loc[i, 'Metadata_Site']}.npz")
        
        if os.path.isfile(filename):
            try:
                with np.load(filename) as data:
                    cells = data["features"]
                    cells_f = cells[~np.isnan(cells).any(axis=1)]
                    
                    if len(cells_f) > 0:
                        if well_id not in well_storage:
                            well_storage[well_id] = []
                            well_to_treatment[well_id] = treatment
                        well_storage[well_id].append(cells_f)
            except:
                continue

    for well_id, feature_list in well_storage.items():
        all_cells_in_well = np.vstack(feature_list)
        well_median = np.median(all_cells_in_well, axis=0)
        
        row = {"Plate": plate_id, "Well_ID": well_id, "Treatment": well_to_treatment[well_id]}
        for idx, val in enumerate(well_median):
            row[idx] = val
        all_plates_data.append(row)

# Combine everything into one giant DataFrame
wells_df = pd.DataFrame(all_plates_data)
print(f"\nAggregation complete. Total Wells: {len(wells_df)}")

# ==========================================
# 2. SAVE TO CSV
# ==========================================
# index=False prevents pandas from writing a column for the row numbers
wells_df.to_csv(OUTPUT_CSV, index=False)
print(f"Successfully saved to: {OUTPUT_CSV}")

In [ ]:
import umap
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text
import matplotlib.cm as cm
import numpy as np

# --- 1. Dynamic Color Setup ---
plate_list = sorted(wells_df['Plate'].unique())
num_plates = len(plate_list)

# Use a qualitative colormap (Set1, Set3, or Dark2 are good for distinct categories)
# 'tab10' or 'tab20' are excellent for up to 10 or 20 distinct categories
cmap = plt.get_cmap('tab10') 
plate_colors = [cmap(i) for i in np.linspace(0, 1, num_plates)]
plate_color_map = {plate: plate_colors[i] for i, plate in enumerate(plate_list)}

# --- 2. Configuration ---
feature_cols = [i for i in range(NUM_CHANNELS * FEATS_PER_CH)]
plot_configs = [
    {"name": "All Channels", "indices": feature_cols},
    {"name": "Channel 1", "indices": list(range(0, 1280))},
    {"name": "Channel 2", "indices": list(range(1280, 2560))},
    {"name": "Channel 3", "indices": list(range(2560, 3840))},
    {"name": "Channel 4", "indices": list(range(3840, 5120))},
    {"name": "Channel 5", "indices": list(range(5120, 6400))}
]

fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    print(f"Running UMAP for {config['name']}...")
    
    X_subset = wells_df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X_subset)
    
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    texts = []
    
    # 3. Plot Plate by Plate
    for plate in plate_list:
        mask = wells_df['Plate'] == plate
        plate_embed = embedding[mask]
        plate_meta = wells_df[mask]
        
        is_ctrl = plate_meta[TREATMENT_COL] == "no_sgRNA"
        
        # Plot Mutants (Filled Circles)
        ax.scatter(plate_embed[~is_ctrl, 0], plate_embed[~is_ctrl, 1], 
                   color=plate_color_map[plate], marker='o', s=60, alpha=0.6, 
                   label=f"{plate} (Mutant)")
        
        # Plot Controls (Open Squares)
        ax.scatter(plate_embed[is_ctrl, 0], plate_embed[is_ctrl, 1], 
                   edgecolors=plate_color_map[plate], facecolors='none', 
                   marker='s', s=130, linewidths=2, label=f"{plate} (Control)")

        # 4. Add labels
        for i, row in enumerate(plate_meta.itertuples()):
            label = f"{row.Treatment}_{row.Plate.split('_')[-1]}"
            texts.append(ax.text(plate_embed[i, 0], plate_embed[i, 1], label, 
                                 fontsize=6, color=plate_color_map[plate]))

    # Formatting
    ax.set_title(f"{config['name']}", fontsize=16, fontweight='bold')
    
    # Move legend to the side if it gets too crowded with 6+ plates
    if idx == 0:
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), markerscale=1, fontsize=8, ncol=1)
    
    # Adjust text to prevent overlapping labels
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.3))

plt.suptitle("Combined UMAP: Multi-Plate Analysis", fontsize=24, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Check if your dataframe and the last embedding still exist
print(wells_df.head())
print(embedding.shape)

In [ ]:
# Run this once to 'pin' the results to your dataframe
# If you already ran UMAP, this loop will 'attach' the math to the rows
for config in plot_configs:
    X_subset = wells_df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X_subset)
    
    # This is the slow part, but once it's done, it's done forever
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Store these so you can change colors 100 times in 1 second
    wells_df[f"umap_{config['name']}_x"] = embedding[:, 0]
    wells_df[f"umap_{config['name']}_y"] = embedding[:, 1]

# SAVE TO DISK so you can restart your PC and keep the math
wells_df.to_pickle("umap_results_backup.pkl")

In [ ]:
# Save to your external drive
wells_df.to_pickle("/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal/umap_results_backup.pkl")

print("File saved successfully to the external drive!")

In [ ]:
# Extract Timepoint from Plate string (e.g., 'PLATE1_T0' -> 'T0')
wells_df['Timepoint'] = wells_df['Plate'].apply(lambda x: x.split('_')[-1])

# Create a unique group name for the legend/coloring
wells_df['Color_Group'] = wells_df['Plate'] + "_" + wells_df['Timepoint']
unique_groups = sorted(wells_df['Color_Group'].unique())

# Set up a colormap with enough colors for all combinations
cmap = plt.get_cmap('tab20') # 'tab20' is great for ~20 distinct categories
colors = [cmap(i) for i in np.linspace(0, 1, len(unique_groups))]
color_map = dict(zip(unique_groups, colors))

fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    x_col = f"umap_{config['name']}_x"
    y_col = f"umap_{config['name']}_y"
    
    texts = []
    for group in unique_groups:
        mask = wells_df['Color_Group'] == group
        subset = wells_df[mask]
        
        is_ctrl = subset['Treatment'] == "no_sgRNA" # Adjust if your control name is different
        
        # Plot Mutants (Circles)
        ax.scatter(subset.loc[~is_ctrl, x_col], subset.loc[~is_ctrl, y_col], 
                   color=color_map[group], marker='o', s=60, alpha=0.6, label=f"{group}")
        
        # Plot Controls (Squares)
        ax.scatter(subset.loc[is_ctrl, x_col], subset.loc[is_ctrl, y_col], 
                   edgecolors=color_map[group], facecolors='none', marker='s', s=130, linewidths=2)

    ax.set_title(config['name'], fontsize=16, fontweight='bold')
    if idx == 0:
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), ncol=2, fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Use raw string (r'') for Windows paths to handle backslashes correctly
# I've updated the path based on your description
file_path = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal/aggregated_wells_data.csv"

# Read only the first 5 rows to see the structure
df_preview = pd.read_csv(file_path, nrows=50)

print(f"File structure for: {file_path}")
print(f"Columns found: {df_preview.columns.tolist()[:10]} ... [and 6393 more]")
display(df_preview.head())

In [ ]:
import pandas as pd
import numpy as np
import os

# 1. Setup Paths
PROJECT_ROOT =  "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")

# 2. Load Raw Data
print("Loading raw aggregated data...")
df_raw = pd.read_csv(INPUT_CSV)

# Identify features (0-6399)
feature_cols = [str(i) for i in range(6400)]
df_raw.columns = [str(c) for c in df_raw.columns]

def select_features_on_raw_data(df, features, anchor_plate='PLATE5_T0', top_n=500):
    # Aggregate Raw Control Medians
    ctrls = df[df['Treatment'] == 'no_sgRNA']
    plate_medians = ctrls.groupby('Plate')[features].median()

    # A. Within-Plate Drift (P5 T0 vs P5 T1)
    # On raw data, this shows how much the machine/environment drifts over time
    within_plate_drift = (plate_medians.loc['PLATE5_T1'] - plate_medians.loc['PLATE5_T0']).abs()

    # B. Across-Plate Drift (P1 T0 vs P5 T0)
    # On raw data, this shows the true technical difference between physical plates
    anchor_t0 = plate_medians.loc['PLATE5_T0']
    other_t0s = [p for p in plate_medians.index if '_T0' in p and p != 'PLATE5_T0']
    
    across_plate_drift = pd.concat([
        (plate_medians.loc[p] - anchor_t0).abs() for p in other_t0s
    ], axis=1).mean(axis=1)

    # C. Total Technical Noise
    total_tech_noise = (within_plate_drift + across_plate_drift) / 2

    # D. Biological Signal (Mutant Impact)
    # How much do mutants deviate from the overall raw control median?
    global_ctrl_median = ctrls[features].median()
    mutant_impact = (df[features] - global_ctrl_median).abs().quantile(0.95)

    # E. Final Quality Score
    # Features where Mutant Impact is high relative to Raw Technical Noise
    quality_score = mutant_impact / (total_tech_noise + 1e-6)

    # Pick top features
    variance_filter = df[features].std() > 0.01
    vetted_list = quality_score[variance_filter].sort_values(ascending=False).head(top_n).index.tolist()
    
    return vetted_list

# --- EXECUTION ---
vetted_cols = select_features_on_raw_data(df_raw, feature_cols, top_n=500)

# Now that we have the features, we can create the filtered dataframe
df_vetted_raw = df_raw[['Plate', 'Well_ID', 'Treatment'] + vetted_cols]

print(f"Features selected! Best feature index: {vetted_cols[0]}")

In [ ]:
def robust_normalize_final(df, features):
    normalized_list = []
    for plate in df['Plate'].unique():
        plate_df = df[df['Plate'] == plate].copy()
        controls = plate_df[plate_df['Treatment'] == 'nosgrna']
        if not controls.empty:
            median = controls[features].median()
            mad = (controls[features] - median).abs().median()
            plate_df[features] = (plate_df[features] - median) / (mad + 1e-6)
        normalized_list.append(plate_df)
    return pd.concat(normalized_list)

df_final_normalized = robust_normalize_final(df_vetted_raw, vetted_cols)

# Save the final masterpiece
df_final_normalized.to_csv(os.path.join(PROJECT_ROOT, "vetted_biological_profiles.csv"), index=False)
print("Final normalized and vetted profiles saved.")

In [ ]:
import pandas as pd
import numpy as np
import umap
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text

# 1. Load your vetted and normalized data
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_biological_profiles.csv")
df = pd.read_csv(file_path)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. Dynamic Color Setup (Every Plate + Timepoint gets a unique color)
# We use 'tab20' which has 20 distinct colors. If you have more, we cycle them.
plate_list = sorted(df['Plate'].unique())
num_plates = len(plate_list)
cmap = plt.get_cmap('tab20') 

# This creates a dictionary where 'PLATE1_T0' has a different color than 'PLATE1_T1'
plate_color_map = {plate: cmap(i % 20) for i, plate in enumerate(plate_list)}

# 3. Define Channel Groups
# We identify which of our 500 vetted features belong to which original channel
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1 ", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# 4. Run the UMAP Grid
fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    if not config['indices']:
        ax.set_title(f"{config['name']} (No features survived vetting)")
        continue
        
    print(f"Running UMAP for {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    texts = []
    
    # Plot Plate-by-Plate (Distinct colors for PLATE1_T0, PLATE1_T1, etc.)
    for plate in plate_list:
        mask = df['Plate'] == plate
        plate_embed = embedding[mask]
        plate_meta = df[mask]
        
        # Identify controls using your string
        is_ctrl = plate_meta['Treatment'] == "no_sgRNA"
        color = plate_color_map[plate]
        
        # Plot Mutants (Filled Circles)
        ax.scatter(plate_embed[~is_ctrl, 0], plate_embed[~is_ctrl, 1], 
                   color=color, marker='o', s=80, alpha=0.6, 
                   label=f"{plate}")
        
        # Plot Controls (Open Squares - larger to stand out)
        ax.scatter(plate_embed[is_ctrl, 0], plate_embed[is_ctrl, 1], 
                   edgecolors=color, facecolors='none', 
                   marker='s', s=180, linewidths=3)

        # 5. Add Labels for Mutants
        for i, row in enumerate(plate_meta.itertuples()):
            if row.Treatment != "nosgrna":
                # Label format: Target_Timepoint
                label = f"{row.Treatment}_{row.Plate.split('_')[-1]}"
                texts.append(ax.text(plate_embed[i, 0], plate_embed[i, 1], label, 
                                     fontsize=8, color=color, fontweight='bold'))

    # Formatting
    ax.set_title(f"{config['name']} ({len(config['indices'])} features)", fontsize=18, fontweight='bold')
    if idx == 0:
        # One legend for the whole figure
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10, ncol=1, title="Plate & Timepoint")
    
    # Adjust text to prevent overlapping labels
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.3))

plt.suptitle("CRISPRi UMAP Analysis: Vetted & Balanced Features", fontsize=28, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "umap_vetted_colored_by_timepoint.png"), dpi=300)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import os
import umap
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")
CONTROL_LABEL = "no_sgRNA" 

# 2. LOAD RAW DATA
df_raw = pd.read_csv(INPUT_CSV)
feature_cols = [str(i) for i in range(6400)]
df_raw.columns = [str(c) for c in df_raw.columns]

def select_features_global_stability(df, features, top_n=500):
    # Get medians for controls per plate
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    # Timepoints to anchor (Across-Plate Consistency)
    timepoints = ['T0', 'T1', 'T2']
    
    # A. Calculate Across-Plate Noise for EACH timepoint
    # (Checking if Plate 1 T0 looks like Plate 5 T0, etc.)
    tp_noises = []
    for tp in timepoints:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std(axis=0)
            tp_noises.append(noise)
    
    global_across_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)

    # B. Within-Plate Drift (ONLY T0 vs T1)
    # Since T2 has biological growth, we only use T0 and T1 to measure technical drift
    if 'PLATE5_T0' in plate_medians.index and 'PLATE5_T1' in plate_medians.index:
        internal_drift = (plate_medians.loc['PLATE5_T1'] - plate_medians.loc['PLATE5_T0']).abs()
    else:
        print("Warning: PLATE5_T0 or T1 missing. Internal drift set to 0.")
        internal_drift = 0

    # C. Combined Technical Penalty
    total_penalty = (global_across_plate_noise + internal_drift) / 2

    # D. Biological Signal (Mutant Impact)
    overall_ctrl_median = ctrls[features].median()
    mutant_impact = (df[features] - overall_ctrl_median).abs().quantile(0.95)

    # E. Quality Score: Impact / (Total Penalty + epsilon)
    quality_score = mutant_impact / (total_penalty + 1e-6)

    # Filter out dead features and pick top 500
    variance_filter = df[features].std() > 0.01
    vetted_list = quality_score[variance_filter].sort_values(ascending=False).head(top_n).index.tolist()
    return vetted_list

# --- EXECUTION ---
vetted_cols = select_features_global_stability(df_raw, feature_cols, top_n=500)

# Step 3: Perform Final Normalization (Per-Plate centering)
def robust_normalize_final(df, features):
    norm_list = []
    for plate in df['Plate'].unique():
        p_df = df[df['Plate'] == plate].copy()
        p_ctrls = p_df[p_df['Treatment'] == CONTROL_LABEL]
        if not p_ctrls.empty:
            med = p_ctrls[features].median()
            mad = (p_ctrls[features] - med).abs().median()
            p_df[features] = (p_df[features] - med) / (mad + 1e-6)
        norm_list.append(p_df)
    return pd.concat(norm_list, ignore_index=True)

df_vetted_norm = robust_normalize_final(df_raw[['Plate', 'Well_ID', 'Treatment'] + vetted_cols], vetted_cols)
df_vetted_norm.to_csv(os.path.join(PROJECT_ROOT, "vetted_multi_timepoint_profiles.csv"), index=False)

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text

# ==========================================
# 1. SETUP AND PLATE SELECTION
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_multi_timepoint_profiles.csv")
df = pd.read_csv(file_path)

# --- ADD YOUR SELECTION HERE ---
# Use None to plot all plates, or a list like ["PLATE5_T0", "PLATE5_T1"]
SELECTED_PLATES = ["PLATE1_T0", "PLATE1_T1","PLATE1_T2","PLATE2_T0", "PLATE2_T1","PLATE2_T2"] 

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# ==========================================
# 2. DYNAMIC COLOR SETUP
# ==========================================
plate_list = sorted(df['Plate'].unique())
num_plates = len(plate_list)
cmap = plt.get_cmap('tab10') if num_plates <= 10 else plt.get_cmap('tab20')

plate_color_map = {plate: cmap(i % 20) for i, plate in enumerate(plate_list)}

# ==========================================
# 3. DEFINE CHANNELS
# ==========================================
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# ==========================================
# 4. RUN UMAP GRID
# ==========================================
fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    if not config['indices']:
        ax.set_title(f"{config['name']} (0 feats)")
        continue
        
    print(f"Running UMAP for {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    texts = []
    
    for plate in plate_list:
        mask = df['Plate'] == plate
        p_embed = embedding[mask]
        p_meta = df[mask]
        
        # Check for control (handling both possible naming conventions)
        is_ctrl = p_meta['Treatment'].isin(["no_sgRNA", "nosgrna"])
        color = plate_color_map[plate]
        
        # Plot Mutants
        ax.scatter(p_embed[~is_ctrl, 0], p_embed[~is_ctrl, 1], 
                   color=color, marker='o', s=80, alpha=0.6, label=f"{plate}")
        
        # Plot Controls
        ax.scatter(p_embed[is_ctrl, 0], p_embed[is_ctrl, 1], 
                   edgecolors=color, facecolors='none', 
                   marker='s', s=180, linewidths=3)

        # Add Labels
        for i, row in enumerate(p_meta.itertuples()):
            if row.Treatment not in ["no_sgRNA", "nosgrna"]:
                label = f"{row.Treatment}_{row.Plate.split('_')[-1]}"
                texts.append(ax.text(p_embed[i, 0], p_embed[i, 1], label, 
                                     fontsize=8, color=color, fontweight='bold'))

    ax.set_title(f"{config['name']} ({len(config['indices'])} feats)", fontsize=18, fontweight='bold')
    if idx == 0:
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10, title="Plate & Timepoint")
    
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.3))

plt.suptitle("CRISPRi UMAP Analysis (Filtered Plates)", fontsize=28, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text

# ==========================================
# 1. SETUP AND PLATE SELECTION
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_multi_timepoint_profiles.csv")
df = pd.read_csv(file_path)

# --- ADD YOUR SELECTION HERE ---
# Use None to plot all plates, or a list like ["PLATE5_T0", "PLATE5_T1"]
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2","PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2","PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# ==========================================
# 2. DYNAMIC COLOR SETUP
# ==========================================
plate_list = sorted(df['Plate'].unique())
num_plates = len(plate_list)
cmap = plt.get_cmap('tab10') if num_plates <= 10 else plt.get_cmap('tab20')

plate_color_map = {plate: cmap(i % 20) for i, plate in enumerate(plate_list)}

# ==========================================
# 3. DEFINE CHANNELS
# ==========================================
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    
]

# ==========================================
# 4. RUN UMAP GRID
# ==========================================
fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    if not config['indices']:
        ax.set_title(f"{config['name']} (0 feats)")
        continue
        
    print(f"Running UMAP for {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    texts = []
    
    for plate in plate_list:
        mask = df['Plate'] == plate
        p_embed = embedding[mask]
        p_meta = df[mask]
        
        # Check for control (handling both possible naming conventions)
        is_ctrl = p_meta['Treatment'].isin(["no_sgRNA", "nosgrna"])
        color = plate_color_map[plate]
        
        # Plot Mutants
        ax.scatter(p_embed[~is_ctrl, 0], p_embed[~is_ctrl, 1], 
                   color=color, marker='o', s=80, alpha=0.6, label=f"{plate}")
        
        # Plot Controls
        ax.scatter(p_embed[is_ctrl, 0], p_embed[is_ctrl, 1], 
                   edgecolors=color, facecolors='none', 
                   marker='s', s=180, linewidths=3)

        # Add Labels
        for i, row in enumerate(p_meta.itertuples()):
            if row.Treatment not in ["no_sgRNA", "nosgrna"]:
                label = f"{row.Treatment}_{row.Plate.split('_')[-1]}"
                texts.append(ax.text(p_embed[i, 0], p_embed[i, 1], label, 
                                     fontsize=8, color=color, fontweight='bold'))

    ax.set_title(f"{config['name']} ({len(config['indices'])} feats)", fontsize=18, fontweight='bold')
    if idx == 0:
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10, title="Plate & Timepoint")
    
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.3))

plt.suptitle("CRISPRi UMAP Analysis (Filtered Plates)", fontsize=28, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import os

# 1. Setup Paths
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")
CONTROL_LABEL = "no_sgRNA"

# 2. Load Raw Data
print("Loading raw aggregated data...")
df_raw = pd.read_csv(INPUT_CSV)

# Identify features (0-6399)
feature_cols = [str(i) for i in range(6400)]
df_raw.columns = [str(c) for c in df_raw.columns]

def select_features_stability_only(df, features, top_n=500):
    # Aggregate Raw Control Medians
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    # --- A. Within-Plate Drift (P5 T0 vs T1) ---
    # Pure technical drift within one plate
    within_plate_drift = (plate_medians.loc['PLATE5_T1'] - plate_medians.loc['PLATE5_T0']).abs()

    # --- B. Across-Plate Drift (Batch Noise per Timepoint) ---
    # Variation between different plates at the same timepoint
    timepoints = ['T0', 'T1', 'T2']
    tp_batch_noises = []

    for tp in timepoints:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            # Measure the standard deviation across plates at this timepoint
            tp_noise = plate_medians.loc[tp_plates].std(axis=0)
            tp_batch_noises.append(tp_noise)
    
    across_plate_drift = pd.concat(tp_batch_noises, axis=1).mean(axis=1)

    # --- C. Total Technical Noise ---
    # We want to minimize this value
    total_tech_noise = (within_plate_drift + across_plate_drift) / 2

    # --- D. Selection ---
    # 1. Filter out features with no variance (completely flat/dead)
    variance_filter = df[features].std() > 0.01
    
    # 2. Rank by lowest noise first
    # We sort ascending because lower noise = higher quality stability
    vetted_list = total_tech_noise[variance_filter].sort_values(ascending=True).head(top_n).index.tolist()
    
    return vetted_list

# --- EXECUTION ---
vetted_cols = select_features_stability_only(df_raw, feature_cols, top_n=500)

# Step 3: Perform Final Normalization (Per-Plate centering)
def robust_normalize_final(df, features):
    norm_list = []
    for plate in df['Plate'].unique():
        p_df = df[df['Plate'] == plate].copy()
        p_ctrls = p_df[p_df['Treatment'] == CONTROL_LABEL]
        if not p_ctrls.empty:
            med = p_ctrls[features].median()
            mad = (p_ctrls[features] - med).abs().median()
            p_df[features] = (p_df[features] - med) / (mad + 1e-6)
        norm_list.append(p_df)
    return pd.concat(norm_list, ignore_index=True)

# Select metadata + the 500 most stable features
df_vetted_raw_subset = df_raw[['Plate', 'Well_ID', 'Treatment'] + vetted_cols]

# Apply the normalization to align the remaining minor offsets
df_vetted_norm = robust_normalize_final(df_vetted_raw_subset, vetted_cols)

# --- SAVE ---
output_filename = "vetted_stability_only_profiles.csv"
df_vetted_norm.to_csv(os.path.join(PROJECT_ROOT, output_filename), index=False)

print(f"Selection based on PURE STABILITY complete.")
print(f"Saved 500 quietest features to: {output_filename}")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text

# ==========================================
# 1. SETUP AND PLATE SELECTION
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_stability_only_profiles.csv")
df = pd.read_csv(file_path)

# --- ADD YOUR SELECTION HERE ---
# Use None to plot all plates, or a list like ["PLATE5_T0", "PLATE5_T1"]
SELECTED_PLATES = ["PLATE1_T0", "PLATE1_T1","PLATE1_T2","PLATE2_T0", "PLATE2_T1","PLATE2_T2"] 

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# ==========================================
# 2. DYNAMIC COLOR SETUP
# ==========================================
plate_list = sorted(df['Plate'].unique())
num_plates = len(plate_list)
cmap = plt.get_cmap('tab10') if num_plates <= 10 else plt.get_cmap('tab20')

plate_color_map = {plate: cmap(i % 20) for i, plate in enumerate(plate_list)}

# ==========================================
# 3. DEFINE CHANNELS
# ==========================================
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# ==========================================
# 4. RUN UMAP GRID
# ==========================================
fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    if not config['indices']:
        ax.set_title(f"{config['name']} (0 feats)")
        continue
        
    print(f"Running UMAP for {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    texts = []
    
    for plate in plate_list:
        mask = df['Plate'] == plate
        p_embed = embedding[mask]
        p_meta = df[mask]
        
        # Check for control (handling both possible naming conventions)
        is_ctrl = p_meta['Treatment'].isin(["no_sgRNA", "nosgrna"])
        color = plate_color_map[plate]
        
        # Plot Mutants
        ax.scatter(p_embed[~is_ctrl, 0], p_embed[~is_ctrl, 1], 
                   color=color, marker='o', s=80, alpha=0.6, label=f"{plate}")
        
        # Plot Controls
        ax.scatter(p_embed[is_ctrl, 0], p_embed[is_ctrl, 1], 
                   edgecolors=color, facecolors='none', 
                   marker='s', s=180, linewidths=3)

        # Add Labels
        for i, row in enumerate(p_meta.itertuples()):
            if row.Treatment not in ["no_sgRNA", "nosgrna"]:
                label = f"{row.Treatment}_{row.Plate.split('_')[-1]}"
                texts.append(ax.text(p_embed[i, 0], p_embed[i, 1], label, 
                                     fontsize=8, color=color, fontweight='bold'))

    ax.set_title(f"{config['name']} ({len(config['indices'])} feats)", fontsize=18, fontweight='bold')
    if idx == 0:
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10, title="Plate & Timepoint")
    
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.3))

plt.suptitle("CRISPRi UMAP Analysis (Filtered Plates)", fontsize=28, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import os

# 1. Setup Paths
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")
CONTROL_LABEL = "no_sgRNA"

# 2. Load Raw Data
print("Loading raw aggregated data...")
df_raw = pd.read_csv(INPUT_CSV)

# Identify features (0-6399)
feature_cols = [str(i) for i in range(6400)]
df_raw.columns = [str(c) for c in df_raw.columns]

def select_features_stability_only(df, features, top_n=500):
    # Aggregate Raw Control Medians
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    # --- A. Within-Plate Drift (P5 T0 vs T1) ---
    # Pure technical drift within one plate
    within_plate_drift = (plate_medians.loc['PLATE5_T1'] - plate_medians.loc['PLATE5_T0']).abs()

    # --- B. Across-Plate Drift (Batch Noise per Timepoint) ---
    # Variation between different plates at the same timepoint
    timepoints = ['T0', 'T1', 'T2']
    tp_batch_noises = []

    for tp in timepoints:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            # Measure the standard deviation across plates at this timepoint
            tp_noise = plate_medians.loc[tp_plates].std(axis=0)
            tp_batch_noises.append(tp_noise)
    
    across_plate_drift = pd.concat(tp_batch_noises, axis=1).mean(axis=1)

    # --- C. Total Technical Noise ---
    # We want to minimize this value
    total_tech_noise = (within_plate_drift + across_plate_drift) / 2

    # --- D. Selection ---
    # 1. Filter out features with no variance (completely flat/dead)
    variance_filter = df[features].std() > 0.01
    
    # 2. Rank by lowest noise first
    # We sort ascending because lower noise = higher quality stability
    vetted_list = total_tech_noise[variance_filter].sort_values(ascending=True).head(top_n).index.tolist()
    
    return vetted_list

# --- EXECUTION ---
vetted_cols = select_features_stability_only(df_raw, feature_cols, top_n=500)



# Select metadata + the 500 most stable features
df_vetted_raw_subset = df_raw[['Plate', 'Well_ID', 'Treatment'] + vetted_cols]

# Apply the normalization to align the remaining minor offsets
df_vetted_norm = df_vetted_raw_subset 
# --- SAVE ---
output_filename = "vetted_stability_only_profiles.csv"
df_vetted_norm.to_csv(os.path.join(PROJECT_ROOT, output_filename), index=False)

print(f"Selection based on PURE STABILITY complete.")
print(f"Saved 500 quietest features to: {output_filename}")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text

# ==========================================
# 1. SETUP AND PLATE SELECTION
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_stability_only_profiles.csv")
df = pd.read_csv(file_path)

# --- ADD YOUR SELECTION HERE ---
# Use None to plot all plates, or a list like ["PLATE5_T0", "PLATE5_T1"]
SELECTED_PLATES = ["PLATE1_T0", "PLATE1_T1","PLATE1_T2","PLATE2_T0", "PLATE2_T1","PLATE2_T2"] 

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# ==========================================
# 2. DYNAMIC COLOR SETUP
# ==========================================
plate_list = sorted(df['Plate'].unique())
num_plates = len(plate_list)
cmap = plt.get_cmap('tab10') if num_plates <= 10 else plt.get_cmap('tab20')

plate_color_map = {plate: cmap(i % 20) for i, plate in enumerate(plate_list)}

# ==========================================
# 3. DEFINE CHANNELS
# ==========================================
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    
]

# ==========================================
# 4. RUN UMAP GRID
# ==========================================
fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    if not config['indices']:
        ax.set_title(f"{config['name']} (0 feats)")
        continue
        
    print(f"Running UMAP for {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    texts = []
    
    for plate in plate_list:
        mask = df['Plate'] == plate
        p_embed = embedding[mask]
        p_meta = df[mask]
        
        # Check for control (handling both possible naming conventions)
        is_ctrl = p_meta['Treatment'].isin(["no_sgRNA", "nosgrna"])
        color = plate_color_map[plate]
        
        # Plot Mutants
        ax.scatter(p_embed[~is_ctrl, 0], p_embed[~is_ctrl, 1], 
                   color=color, marker='o', s=80, alpha=0.6, label=f"{plate}")
        
        # Plot Controls
        ax.scatter(p_embed[is_ctrl, 0], p_embed[is_ctrl, 1], 
                   edgecolors=color, facecolors='none', 
                   marker='s', s=180, linewidths=3)

        # Add Labels
        for i, row in enumerate(p_meta.itertuples()):
            if row.Treatment not in ["no_sgRNA", "nosgrna"]:
                label = f"{row.Treatment}_{row.Plate.split('_')[-1]}"
                texts.append(ax.text(p_embed[i, 0], p_embed[i, 1], label, 
                                     fontsize=8, color=color, fontweight='bold'))

    ax.set_title(f"{config['name']} ({len(config['indices'])} feats)", fontsize=18, fontweight='bold')
    if idx == 0:
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10, title="Plate & Timepoint")
    
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.3))

plt.suptitle("CRISPRi UMAP Analysis (Filtered Plates)", fontsize=28, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import os

# --- 1. SETUP ---
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")

# SET THESE TO MATCH YOUR CSV EXACTLY
CONTROL_LABEL = "no_sgRNA" 
TARGETS = ['pbpB', 'ftsZ', 'ftsL', 'leuS', 'metS', 'cmk', 'tagO']

# --- 2. LOAD ---
df_raw = pd.read_csv(INPUT_CSV)
feature_cols = [str(i) for i in range(6400)]
df_raw.columns = [str(c) for c in df_raw.columns]

def select_features_refined_phenotype(df, features, top_n=500):
    # DIAGNOSTIC: Print available labels to catch typos
    available_treatments = df['Treatment'].unique()
    print(f"Treatments found in CSV: {available_treatments[:10]}...")
    
    # Aggregate medians by Plate and Treatment
    grouped = df.groupby(['Plate', 'Treatment'])[features].median()

    # --- PART 1: THE PENALTY (Noise & Technical Drift) ---
    
    # A. Across-Plate Variation (T0, T1, T2)
    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in grouped.index.get_level_values(0).unique() if p.endswith(tp)]
        if len(tp_plates) > 1:
            try:
                # Calculate variation across plates for the control
                noise = grouped.loc[(tp_plates, CONTROL_LABEL), :].std()
                tp_batch_noises.append(noise)
            except KeyError: continue
    across_plate_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1) if tp_batch_noises else pd.Series(0, index=features)

    # B. Global Temporal Drift (T0 vs T1 for ALL plates)
    base_names = set([p.split('_')[0] for p in grouped.index.get_level_values(0)])
    temporal_drifts = []
    for base in base_names:
        p0, p1 = f"{base}_T0", f"{base}_T1"
        if p0 in grouped.index.get_level_values(0) and p1 in grouped.index.get_level_values(0):
            try:
                drift = (grouped.loc[(p1, CONTROL_LABEL)] - grouped.loc[(p0, CONTROL_LABEL)]).abs()
                temporal_drifts.append(drift)
            except KeyError: continue
    global_temporal_drift = pd.concat(temporal_drifts, axis=1).mean(axis=1) if temporal_drifts else pd.Series(0, index=features)

    # C. Early Effect Penalty (Mutant vs Control at T0)
    t0_plates = [p for p in grouped.index.get_level_values(0).unique() if p.endswith('T0')]
    t0_deviations = []
    for target in TARGETS:
        try:
            target_t0 = grouped.xs(target, level=1).loc[t0_plates]
            ctrl_t0 = grouped.xs(CONTROL_LABEL, level=1).loc[t0_plates]
            t0_deviations.append((target_t0 - ctrl_t0).abs().mean())
        except (KeyError, ValueError): continue
    early_effect_penalty = pd.concat(t0_deviations, axis=1).mean(axis=1) if t0_deviations else pd.Series(0, index=features)

    total_penalty = across_plate_noise + global_temporal_drift + early_effect_penalty

    # --- PART 2: THE SIGNAL (Biology at T2) ---
    t2_plates = [p for p in grouped.index.get_level_values(0).unique() if p.endswith('T2')]
    t2_signals = []
    for target in TARGETS:
        try:
            target_t2 = grouped.xs(target, level=1).loc[t2_plates]
            ctrl_t2 = grouped.xs(CONTROL_LABEL, level=1).loc[t2_plates]
            t2_signals.append((target_t2 - ctrl_t2).abs().mean())
        except (KeyError, ValueError): continue
    t2_biological_signal = pd.concat(t2_signals, axis=1).mean(axis=1) if t2_signals else pd.Series(0, index=features)

    # --- PART 3: SCORING ---
    # quality = T2 biological signal / (all technical noise + early mutant bias)
    quality_score = t2_biological_signal / (total_penalty + 1e-6)
    
    variance_filter = df[features].std() > 0.01
    vetted_list = quality_score[variance_filter].sort_values(ascending=False).head(top_n).index.tolist()
    return vetted_list

# --- EXECUTION ---
vetted_cols = select_features_refined_phenotype(df_raw, feature_cols, top_n=500)

if not vetted_cols:
    print("Selection failed. Check diagnostic output for label mismatches.")
else:
    # Normalize and save
    def robust_normalize(df, features):
        norm_list = []
        for plate in df['Plate'].unique():
            p_df = df[df['Plate'] == plate].copy()
            p_ctrls = p_df[p_df['Treatment'] == CONTROL_LABEL]
            if not p_ctrls.empty:
                med = p_ctrls[features].median()
                mad = (p_ctrls[features] - med).abs().median()
                p_df[features] = (p_df[features] - med) / (mad + 1e-6)
            norm_list.append(p_df)
        return pd.concat(norm_list, ignore_index=True)

    df_final = robust_normalize(df_raw[['Plate', 'Well_ID', 'Treatment'] + vetted_cols], vetted_cols)
    df_final.to_csv(os.path.join(PROJECT_ROOT, "vetted_phenotype_final.csv"), index=False)
    print(f"Final vetting complete. {len(vetted_cols)} features saved.")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text

# ==========================================
# 1. SETUP AND PLATE SELECTION
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_phenotype_final.csv")
df = pd.read_csv(file_path)

# --- ADD YOUR SELECTION HERE ---
# Use None to plot all plates, or a list like ["PLATE5_T0", "PLATE5_T1"]
SELECTED_PLATES = ["PLATE1_T0", "PLATE1_T1","PLATE1_T2","PLATE2_T0", "PLATE2_T1","PLATE2_T2"] 

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# ==========================================
# 2. DYNAMIC COLOR SETUP
# ==========================================
plate_list = sorted(df['Plate'].unique())
num_plates = len(plate_list)
cmap = plt.get_cmap('tab10') if num_plates <= 10 else plt.get_cmap('tab20')

plate_color_map = {plate: cmap(i % 20) for i, plate in enumerate(plate_list)}

# ==========================================
# 3. DEFINE CHANNELS
# ==========================================
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    
]

# ==========================================
# 4. RUN UMAP GRID
# ==========================================
fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    if not config['indices']:
        ax.set_title(f"{config['name']} (0 feats)")
        continue
        
    print(f"Running UMAP for {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    texts = []
    
    for plate in plate_list:
        mask = df['Plate'] == plate
        p_embed = embedding[mask]
        p_meta = df[mask]
        
        # Check for control (handling both possible naming conventions)
        is_ctrl = p_meta['Treatment'].isin(["no_sgRNA", "nosgrna"])
        color = plate_color_map[plate]
        
        # Plot Mutants
        ax.scatter(p_embed[~is_ctrl, 0], p_embed[~is_ctrl, 1], 
                   color=color, marker='o', s=80, alpha=0.6, label=f"{plate}")
        
        # Plot Controls
        ax.scatter(p_embed[is_ctrl, 0], p_embed[is_ctrl, 1], 
                   edgecolors=color, facecolors='none', 
                   marker='s', s=180, linewidths=3)

        # Add Labels
        for i, row in enumerate(p_meta.itertuples()):
            if row.Treatment not in ["no_sgRNA", "nosgrna"]:
                label = f"{row.Treatment}_{row.Plate.split('_')[-1]}"
                texts.append(ax.text(p_embed[i, 0], p_embed[i, 1], label, 
                                     fontsize=8, color=color, fontweight='bold'))

    ax.set_title(f"{config['name']} ({len(config['indices'])} feats)", fontsize=18, fontweight='bold')
    if idx == 0:
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10, title="Plate & Timepoint")
    
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.3))

plt.suptitle("CRISPRi UMAP Analysis (Filtered Plates)", fontsize=28, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
#step by step

In [ ]:
import pandas as pd
import numpy as np
import os

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")
CONTROL_LABEL = "no_sgRNA" 

# 2. LOAD
df_raw = pd.read_csv(INPUT_CSV)
feature_cols = [str(i) for i in range(6400)]
df_raw.columns = [str(c) for c in df_raw.columns]

def filter_step1_across_plate_stability(df, features, top_n_to_keep=2000):
    # Get medians for controls per plate
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    # Calculate variation across plates for each timepoint
    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        # Find all physical plates for this timepoint (e.g., PLATE1_T0, PLATE2_T0...)
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        
        if len(tp_plates) > 1:
            # Standard deviation across plates at this specific timepoint
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
            print(f"Calculated batch noise for {tp} using {len(tp_plates)} plates.")
    
    # Average the noise across the timepoints
    # Features with high values here are sensitive to the physical plate/batch
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)

    # Filter: Keep only the features with the LOWEST batch noise
    # Also ensure the feature isn't just constant zeros
    variance_filter = df[features].std() > 0.01
    stable_features = total_batch_noise[variance_filter].sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    
    return stable_features, total_batch_noise

# EXECUTE STEP 1
stable_cols, noise_scores = filter_step1_across_plate_stability(df_raw, feature_cols, top_n_to_keep=2000)

# Create a temporary dataframe with just these stable features
df_step1 = df_raw[['Plate', 'Well_ID', 'Treatment'] + stable_cols]

# Save progress
df_step1.to_csv(os.path.join(PROJECT_ROOT, "step1_batch_stable_profiles.csv"), index=False)
print(f"Step 1 Complete: Kept {len(stable_cols)} features with the lowest across-plate variation.")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text

# ==========================================
# 1. SETUP AND PLATE SELECTION
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "step1_batch_stable_profiles.csv")
df = pd.read_csv(file_path)

# --- ADD YOUR SELECTION HERE ---
# Use None to plot all plates, or a list like ["PLATE5_T0", "PLATE5_T1"]
SELECTED_PLATES = ["PLATE1_T0", "PLATE1_T1","PLATE1_T2","PLATE2_T0", "PLATE2_T1","PLATE2_T2"] 

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# ==========================================
# 2. DYNAMIC COLOR SETUP
# ==========================================
plate_list = sorted(df['Plate'].unique())
num_plates = len(plate_list)
cmap = plt.get_cmap('tab10') if num_plates <= 10 else plt.get_cmap('tab20')

plate_color_map = {plate: cmap(i % 20) for i, plate in enumerate(plate_list)}

# ==========================================
# 3. DEFINE CHANNELS
# ==========================================
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    
]

# ==========================================
# 4. RUN UMAP GRID
# ==========================================
fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    if not config['indices']:
        ax.set_title(f"{config['name']} (0 feats)")
        continue
        
    print(f"Running UMAP for {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    texts = []
    
    for plate in plate_list:
        mask = df['Plate'] == plate
        p_embed = embedding[mask]
        p_meta = df[mask]
        
        # Check for control (handling both possible naming conventions)
        is_ctrl = p_meta['Treatment'].isin(["no_sgRNA", "nosgrna"])
        color = plate_color_map[plate]
        
        # Plot Mutants
        ax.scatter(p_embed[~is_ctrl, 0], p_embed[~is_ctrl, 1], 
                   color=color, marker='o', s=80, alpha=0.6, label=f"{plate}")
        
        # Plot Controls
        ax.scatter(p_embed[is_ctrl, 0], p_embed[is_ctrl, 1], 
                   edgecolors=color, facecolors='none', 
                   marker='s', s=180, linewidths=3)

        # Add Labels
        for i, row in enumerate(p_meta.itertuples()):
            if row.Treatment not in ["no_sgRNA", "nosgrna"]:
                label = f"{row.Treatment}_{row.Plate.split('_')[-1]}"
                texts.append(ax.text(p_embed[i, 0], p_embed[i, 1], label, 
                                     fontsize=8, color=color, fontweight='bold'))

    ax.set_title(f"{config['name']} ({len(config['indices'])} feats)", fontsize=18, fontweight='bold')
    if idx == 0:
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10, title="Plate & Timepoint")
    
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.3))

plt.suptitle("CRISPRi UMAP Analysis (Filtered Plates)", fontsize=28, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
#remove featruse that differ between control andn mutants at T0

In [ ]:
import pandas as pd
import numpy as np
import os

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
# We load the results from Step 1
INPUT_CSV = os.path.join(PROJECT_ROOT, "step1_batch_stable_profiles.csv")
CONTROL_LABEL = "no_sgRNA"

# 2. LOAD STEP 1 DATA
df_step1 = pd.read_csv(INPUT_CSV)
# Identify which columns are features (numeric names)
step1_features = [c for c in df_step1.columns if c.isdigit()]

def filter_step2_t0_neutrality(df, features, top_n_to_keep=1000):
    # Filter for only T0 data
    t0_df = df[df['Plate'].str.endswith('T0')].copy()
    
    # Calculate the median of the controls at T0 across all plates
    t0_ctrl_median = t0_df[t0_df['Treatment'] == CONTROL_LABEL][features].median()
    
    # Calculate how much ALL mutants at T0 deviate from the T0 control
    # We want this value to be as LOW as possible (Neutrality)
    t0_mutants = t0_df[t0_df['Treatment'] != CONTROL_LABEL]
    
    # Calculate the average absolute deviation for every feature at T0
    # Features that 'see' a difference here are problematic
    t0_bias_score = (t0_mutants[features] - t0_ctrl_median).abs().mean()
    
    # Rank by LOWEST bias (most neutral at T0)
    # We keep the top N features that show the least difference at the start
    stable_at_start = t0_bias_score.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    
    print(f"Average T0 bias of selected features: {t0_bias_score[stable_at_start].mean():.4f}")
    print(f"Average T0 bias of discarded features: {t0_bias_score.drop(stable_at_start).mean():.4f}")
    
    return stable_at_start

# EXECUTE STEP 2
neutral_features = filter_step2_t0_neutrality(df_step1, step1_features, top_n_to_keep=1000)

# Create the filtered dataframe
df_step2 = df_step1[['Plate', 'Well_ID', 'Treatment'] + neutral_features]

# Save progress
df_step2.to_csv(os.path.join(PROJECT_ROOT, "step2_t0_neutral_profiles.csv"), index=False)
print(f"Step 2 Complete: Kept {len(neutral_features)} features that are neutral at T0.")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text

# ==========================================
# 1. SETUP AND PLATE SELECTION
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "step2_t0_neutral_profiles.csv")
df = pd.read_csv(file_path)

# --- ADD YOUR SELECTION HERE ---
# Use None to plot all plates, or a list like ["PLATE5_T0", "PLATE5_T1"]
SELECTED_PLATES = ["PLATE1_T0", "PLATE1_T1","PLATE1_T2","PLATE2_T0", "PLATE2_T1","PLATE2_T2"] 

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# ==========================================
# 2. DYNAMIC COLOR SETUP
# ==========================================
plate_list = sorted(df['Plate'].unique())
num_plates = len(plate_list)
cmap = plt.get_cmap('tab10') if num_plates <= 10 else plt.get_cmap('tab20')

plate_color_map = {plate: cmap(i % 20) for i, plate in enumerate(plate_list)}

# ==========================================
# 3. DEFINE CHANNELS
# ==========================================
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    
]

# ==========================================
# 4. RUN UMAP GRID
# ==========================================
fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    if not config['indices']:
        ax.set_title(f"{config['name']} (0 feats)")
        continue
        
    print(f"Running UMAP for {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    texts = []
    
    for plate in plate_list:
        mask = df['Plate'] == plate
        p_embed = embedding[mask]
        p_meta = df[mask]
        
        # Check for control (handling both possible naming conventions)
        is_ctrl = p_meta['Treatment'].isin(["no_sgRNA", "nosgrna"])
        color = plate_color_map[plate]
        
        # Plot Mutants
        ax.scatter(p_embed[~is_ctrl, 0], p_embed[~is_ctrl, 1], 
                   color=color, marker='o', s=80, alpha=0.6, label=f"{plate}")
        
        # Plot Controls
        ax.scatter(p_embed[is_ctrl, 0], p_embed[is_ctrl, 1], 
                   edgecolors=color, facecolors='none', 
                   marker='s', s=180, linewidths=3)

        # Add Labels
        for i, row in enumerate(p_meta.itertuples()):
            if row.Treatment not in ["no_sgRNA", "nosgrna"]:
                label = f"{row.Treatment}_{row.Plate.split('_')[-1]}"
                texts.append(ax.text(p_embed[i, 0], p_embed[i, 1], label, 
                                     fontsize=8, color=color, fontweight='bold'))

    ax.set_title(f"{config['name']} ({len(config['indices'])} feats)", fontsize=18, fontweight='bold')
    if idx == 0:
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10, title="Plate & Timepoint")
    
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.3))

plt.suptitle("CRISPRi UMAP Analysis (Filtered Plates)", fontsize=28, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import os

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
# Load the results from Step 2
INPUT_CSV = os.path.join(PROJECT_ROOT, "step2_t0_neutral_profiles.csv")
CONTROL_LABEL = "no_sgRNA"

# 2. LOAD STEP 2 DATA
df_step2 = pd.read_csv(INPUT_CSV)
step2_features = [c for c in df_step2.columns if c.isdigit()]

def filter_step3_temporal_stability(df, features, top_n_to_keep=500):
    # Get medians for controls per plate
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()
    
    # Identify physical plates that have both T0 and T1 data
    base_plate_names = set([p.split('_')[0] for p in plate_medians.index])
    
    temporal_drifts = []
    for base in base_plate_names:
        p0, p1 = f"{base}_T0", f"{base}_T1"
        if p0 in plate_medians.index and p1 in plate_medians.index:
            # Measure the absolute drift between T0 and T1 for this specific plate
            drift = (plate_medians.loc[p1] - plate_medians.loc[p0]).abs()
            temporal_drifts.append(drift)
    
    if not temporal_drifts:
        print("Error: No matching T0 and T1 plates found to calculate drift.")
        return features[:top_n_to_keep]

    # Average the drift across all plates
    avg_temporal_drift = pd.concat(temporal_drifts, axis=1).mean(axis=1)
    
    # Rank by LOWEST drift (most stable over time)
    stable_over_time = avg_temporal_drift.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    
    print(f"Average drift of selected features: {avg_temporal_drift[stable_over_time].mean():.4f}")
    print(f"Average drift of discarded features: {avg_temporal_drift.drop(stable_over_time).mean():.4f}")
    
    return stable_over_time

# EXECUTE STEP 3
stable_time_features = filter_step3_temporal_stability(df_step2, step2_features, top_n_to_keep=500)

# Create the filtered dataframe
df_step3 = df_step2[['Plate', 'Well_ID', 'Treatment'] + stable_time_features]

# Save progress
df_step3.to_csv(os.path.join(PROJECT_ROOT, "step3_temporally_stable_profiles.csv"), index=False)
print(f"Step 3 Complete: Kept {len(stable_time_features)} features with minimal T0-T1 drift.")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text

# ==========================================
# 1. SETUP AND PLATE SELECTION
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "step3_temporally_stable_profiles.csv")
df = pd.read_csv(file_path)

# --- ADD YOUR SELECTION HERE ---
# Use None to plot all plates, or a list like ["PLATE5_T0", "PLATE5_T1"]
SELECTED_PLATES = ["PLATE1_T0", "PLATE1_T1","PLATE1_T2","PLATE2_T0", "PLATE2_T1","PLATE2_T2"] 

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# ==========================================
# 2. DYNAMIC COLOR SETUP
# ==========================================
plate_list = sorted(df['Plate'].unique())
num_plates = len(plate_list)
cmap = plt.get_cmap('tab10') if num_plates <= 10 else plt.get_cmap('tab20')

plate_color_map = {plate: cmap(i % 20) for i, plate in enumerate(plate_list)}

# ==========================================
# 3. DEFINE CHANNELS
# ==========================================
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    
]

# ==========================================
# 4. RUN UMAP GRID
# ==========================================
fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    if not config['indices']:
        ax.set_title(f"{config['name']} (0 feats)")
        continue
        
    print(f"Running UMAP for {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    texts = []
    
    for plate in plate_list:
        mask = df['Plate'] == plate
        p_embed = embedding[mask]
        p_meta = df[mask]
        
        # Check for control (handling both possible naming conventions)
        is_ctrl = p_meta['Treatment'].isin(["no_sgRNA", "nosgrna"])
        color = plate_color_map[plate]
        
        # Plot Mutants
        ax.scatter(p_embed[~is_ctrl, 0], p_embed[~is_ctrl, 1], 
                   color=color, marker='o', s=80, alpha=0.6, label=f"{plate}")
        
        # Plot Controls
        ax.scatter(p_embed[is_ctrl, 0], p_embed[is_ctrl, 1], 
                   edgecolors=color, facecolors='none', 
                   marker='s', s=180, linewidths=3)

        # Add Labels
        for i, row in enumerate(p_meta.itertuples()):
            if row.Treatment not in ["no_sgRNA", "nosgrna"]:
                label = f"{row.Treatment}_{row.Plate.split('_')[-1]}"
                texts.append(ax.text(p_embed[i, 0], p_embed[i, 1], label, 
                                     fontsize=8, color=color, fontweight='bold'))

    ax.set_title(f"{config['name']} ({len(config['indices'])} feats)", fontsize=18, fontweight='bold')
    if idx == 0:
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10, title="Plate & Timepoint")
    
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.3))

plt.suptitle("CRISPRi UMAP Analysis (Filtered Plates)", fontsize=28, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import os

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
# Load the results from your current Step 3 to refine them even further
INPUT_CSV = os.path.join(PROJECT_ROOT, "step3_temporally_stable_profiles.csv")
CONTROL_LABEL = "no_sgRNA"

# 2. LOAD DATA
df_step3 = pd.read_csv(INPUT_CSV)
step3_features = [c for c in df_step3.columns if c.isdigit()]

def filter_step4_aggressive_plate_stability(df, features, top_n_to_keep=300):
    # Aggregate medians for controls per plate
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    # Calculate variation across plates for each timepoint
    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        # Find all physical plates for this timepoint
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        
        if len(tp_plates) > 1:
            # Measure standard deviation across these plates
            # High SD = High Technical Noise for that specific feature
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    # Average the noise across T0, T1, and T2
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)

    # Rank by LOWEST noise (the most stable features across physical plates)
    ultra_stable_features = total_batch_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    
    print(f"Mean Across-Plate Noise (Keep): {total_batch_noise[ultra_stable_features].mean():.4f}")
    print(f"Mean Across-Plate Noise (Drop): {total_batch_noise.drop(ultra_stable_features).mean():.4f}")
    
    return ultra_stable_features

# EXECUTE STEP 4
# We reduce the feature set even further (e.g., down to 300)
ultra_stable_cols = filter_step4_aggressive_plate_stability(df_step3, step3_features, top_n_to_keep=300)

# Create the filtered dataframe
df_step4 = df_step3[['Plate', 'Well_ID', 'Treatment'] + ultra_stable_cols]

# Save progress
df_step4.to_csv(os.path.join(PROJECT_ROOT, "step4_ultra_stable_profiles.csv"), index=False)
print(f"Step 4 Complete: Kept the {len(ultra_stable_cols)} quietest features across all plates.")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text

# ==========================================
# 1. SETUP AND PLATE SELECTION
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "step4_ultra_stable_profiles.csv")
df = pd.read_csv(file_path)

# --- ADD YOUR SELECTION HERE ---
# Use None to plot all plates, or a list like ["PLATE5_T0", "PLATE5_T1"]
SELECTED_PLATES = ["PLATE1_T0", "PLATE1_T1","PLATE1_T2","PLATE2_T0", "PLATE2_T1","PLATE2_T2"] 

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# ==========================================
# 2. DYNAMIC COLOR SETUP
# ==========================================
plate_list = sorted(df['Plate'].unique())
num_plates = len(plate_list)
cmap = plt.get_cmap('tab10') if num_plates <= 10 else plt.get_cmap('tab20')

plate_color_map = {plate: cmap(i % 20) for i, plate in enumerate(plate_list)}

# ==========================================
# 3. DEFINE CHANNELS
# ==========================================
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    
]

# ==========================================
# 4. RUN UMAP GRID
# ==========================================
fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    if not config['indices']:
        ax.set_title(f"{config['name']} (0 feats)")
        continue
        
    print(f"Running UMAP for {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    texts = []
    
    for plate in plate_list:
        mask = df['Plate'] == plate
        p_embed = embedding[mask]
        p_meta = df[mask]
        
        # Check for control (handling both possible naming conventions)
        is_ctrl = p_meta['Treatment'].isin(["no_sgRNA", "nosgrna"])
        color = plate_color_map[plate]
        
        # Plot Mutants
        ax.scatter(p_embed[~is_ctrl, 0], p_embed[~is_ctrl, 1], 
                   color=color, marker='o', s=80, alpha=0.6, label=f"{plate}")
        
        # Plot Controls
        ax.scatter(p_embed[is_ctrl, 0], p_embed[is_ctrl, 1], 
                   edgecolors=color, facecolors='none', 
                   marker='s', s=180, linewidths=3)

        # Add Labels
        for i, row in enumerate(p_meta.itertuples()):
            if row.Treatment not in ["no_sgRNA", "nosgrna"]:
                label = f"{row.Treatment}_{row.Plate.split('_')[-1]}"
                texts.append(ax.text(p_embed[i, 0], p_embed[i, 1], label, 
                                     fontsize=8, color=color, fontweight='bold'))

    ax.set_title(f"{config['name']} ({len(config['indices'])} feats)", fontsize=18, fontweight='bold')
    if idx == 0:
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10, title="Plate & Timepoint")
    
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.3))

plt.suptitle("CRISPRi UMAP Analysis (Filtered Plates)", fontsize=28, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text

# ==========================================
# 1. SETUP AND PLATE SELECTION
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "step4_ultra_stable_profiles.csv")
df = pd.read_csv(file_path)

# --- ADD YOUR SELECTION HERE ---
# Use None to plot all plates, or a list like ["PLATE5_T0", "PLATE5_T1"]
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2","PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2","PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# ==========================================
# 2. DYNAMIC COLOR SETUP
# ==========================================
plate_list = sorted(df['Plate'].unique())
num_plates = len(plate_list)
cmap = plt.get_cmap('tab10') if num_plates <= 10 else plt.get_cmap('tab20')

plate_color_map = {plate: cmap(i % 20) for i, plate in enumerate(plate_list)}

# ==========================================
# 3. DEFINE CHANNELS
# ==========================================
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    
]

# ==========================================
# 4. RUN UMAP GRID
# ==========================================
fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    if not config['indices']:
        ax.set_title(f"{config['name']} (0 feats)")
        continue
        
    print(f"Running UMAP for {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    texts = []
    
    for plate in plate_list:
        mask = df['Plate'] == plate
        p_embed = embedding[mask]
        p_meta = df[mask]
        
        # Check for control (handling both possible naming conventions)
        is_ctrl = p_meta['Treatment'].isin(["no_sgRNA", "nosgrna"])
        color = plate_color_map[plate]
        
        # Plot Mutants
        ax.scatter(p_embed[~is_ctrl, 0], p_embed[~is_ctrl, 1], 
                   color=color, marker='o', s=80, alpha=0.6, label=f"{plate}")
        
        # Plot Controls
        ax.scatter(p_embed[is_ctrl, 0], p_embed[is_ctrl, 1], 
                   edgecolors=color, facecolors='none', 
                   marker='s', s=180, linewidths=3)

        # Add Labels
        for i, row in enumerate(p_meta.itertuples()):
            if row.Treatment not in ["no_sgRNA", "nosgrna"]:
                label = f"{row.Treatment}_{row.Plate.split('_')[-1]}"
                texts.append(ax.text(p_embed[i, 0], p_embed[i, 1], label, 
                                     fontsize=8, color=color, fontweight='bold'))

    ax.set_title(f"{config['name']} ({len(config['indices'])} feats)", fontsize=18, fontweight='bold')
    if idx == 0:
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10, title="Plate & Timepoint")
    
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.3))

plt.suptitle("CRISPRi UMAP Analysis (Filtered Plates)", fontsize=28, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import os

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
# Load the output from Step 3
INPUT_CSV = os.path.join(PROJECT_ROOT, "step3_temporally_stable_profiles.csv")
CONTROL_LABEL = "no_sgRNA"

# 2. LOAD
df_step3 = pd.read_csv(INPUT_CSV)
step3_features = [c for c in df_step3.columns if c.isdigit()]

def filter_step4_plate_convergence(df, features, top_n_to_keep=250):
    # Calculate control medians per plate
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    # Calculate standard deviation across plates at each timepoint
    # This is the "Plate-to-Plate" noise metric
    tp_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            # Measure how much the plates disagree for each feature
            noise = plate_medians.loc[tp_plates].std()
            tp_noises.append(noise)
    
    # Average noise across the experiment
    avg_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)

    # Rank by LOWEST noise (the most "Plate-Blind" features)
    plate_blind_features = avg_plate_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    
    print(f"Mean Plate-to-Plate Noise (Kept): {avg_plate_noise[plate_blind_features].mean():.4f}")
    print(f"Mean Plate-to-Plate Noise (Dropped): {avg_plate_noise.drop(plate_blind_features).mean():.4f}")
    
    return plate_blind_features

# EXECUTE STEP 4
# Pruning down to the 250 most stable features
final_vetted_features = filter_step4_plate_convergence(df_step3, step3_features, top_n_to_keep=200)

# Final Dataframe
df_final = df_step3[['Plate', 'Well_ID', 'Treatment'] + final_vetted_features]

# Save the final vetted list
df_final.to_csv(os.path.join(PROJECT_ROOT, "step4_final_vetted_profiles.csv"), index=False)
print(f"Step 4 Complete: 200 features selected for maximum plate convergence.")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text

# ==========================================
# 1. SETUP AND PLATE SELECTION
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "step4_final_vetted_profiles.csv")
df = pd.read_csv(file_path)

# --- ADD YOUR SELECTION HERE ---
# Use None to plot all plates, or a list like ["PLATE5_T0", "PLATE5_T1"]
SELECTED_PLATES = ["PLATE1_T0", "PLATE1_T1","PLATE1_T2","PLATE2_T0", "PLATE2_T1","PLATE2_T2"] 

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# ==========================================
# 2. DYNAMIC COLOR SETUP
# ==========================================
plate_list = sorted(df['Plate'].unique())
num_plates = len(plate_list)
cmap = plt.get_cmap('tab10') if num_plates <= 10 else plt.get_cmap('tab20')

plate_color_map = {plate: cmap(i % 20) for i, plate in enumerate(plate_list)}

# ==========================================
# 3. DEFINE CHANNELS
# ==========================================
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    
]

# ==========================================
# 4. RUN UMAP GRID
# ==========================================
fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    if not config['indices']:
        ax.set_title(f"{config['name']} (0 feats)")
        continue
        
    print(f"Running UMAP for {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    texts = []
    
    for plate in plate_list:
        mask = df['Plate'] == plate
        p_embed = embedding[mask]
        p_meta = df[mask]
        
        # Check for control (handling both possible naming conventions)
        is_ctrl = p_meta['Treatment'].isin(["no_sgRNA", "nosgrna"])
        color = plate_color_map[plate]
        
        # Plot Mutants
        ax.scatter(p_embed[~is_ctrl, 0], p_embed[~is_ctrl, 1], 
                   color=color, marker='o', s=80, alpha=0.6, label=f"{plate}")
        
        # Plot Controls
        ax.scatter(p_embed[is_ctrl, 0], p_embed[is_ctrl, 1], 
                   edgecolors=color, facecolors='none', 
                   marker='s', s=180, linewidths=3)

        # Add Labels
        for i, row in enumerate(p_meta.itertuples()):
            if row.Treatment not in ["no_sgRNA", "nosgrna"]:
                label = f"{row.Treatment}_{row.Plate.split('_')[-1]}"
                texts.append(ax.text(p_embed[i, 0], p_embed[i, 1], label, 
                                     fontsize=8, color=color, fontweight='bold'))

    ax.set_title(f"{config['name']} ({len(config['indices'])} feats)", fontsize=18, fontweight='bold')
    if idx == 0:
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10, title="Plate & Timepoint")
    
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.3))

plt.suptitle("CRISPRi UMAP Analysis (Filtered Plates)", fontsize=28, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import os

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
# Load the output from Step 4
INPUT_CSV = os.path.join(PROJECT_ROOT, "step4_final_vetted_profiles.csv")

# 2. LOAD
df_step4 = pd.read_csv(INPUT_CSV)
step4_features = [c for c in df_step4.columns if c.isdigit()]

def filter_step5_redundancy(df, features, correlation_threshold=0.9):
    # Calculate the correlation matrix for the vetted features
    corr_matrix = df[features].corr().abs()

    # Select upper triangle of correlation matrix
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    # Find features with correlation greater than the threshold
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    
    # Features to keep
    final_features = [f for f in features if f not in to_drop]
    
    print(f"Redundancy Filter: Identified {len(to_drop)} highly correlated features.")
    print(f"Features remaining: {len(final_features)}")
    
    return final_features

# EXECUTE STEP 5
# Adjust threshold to 0.85 or 0.9 depending on how much you want to prune
non_redundant_features = filter_step5_redundancy(df_step4, step4_features, correlation_threshold=0.9)

# Final Dataframe
df_final_vetted = df_step4[['Plate', 'Well_ID', 'Treatment'] + non_redundant_features]

# --- FINAL SAVE ---
output_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles.csv")
df_final_vetted.to_csv(output_path, index=False)

print(f"Workflow Complete. Master profiles saved to: {output_path}")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from adjustText import adjust_text

# ==========================================
# 1. SETUP AND PLATE SELECTION
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles.csv")
df = pd.read_csv(file_path)

# --- ADD YOUR SELECTION HERE ---
# Use None to plot all plates, or a list like ["PLATE5_T0", "PLATE5_T1"]
SELECTED_PLATES = ["PLATE1_T0", "PLATE1_T1","PLATE1_T2","PLATE2_T0", "PLATE2_T1","PLATE2_T2"] 

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# ==========================================
# 2. DYNAMIC COLOR SETUP
# ==========================================
plate_list = sorted(df['Plate'].unique())
num_plates = len(plate_list)
cmap = plt.get_cmap('tab10') if num_plates <= 10 else plt.get_cmap('tab20')

plate_color_map = {plate: cmap(i % 20) for i, plate in enumerate(plate_list)}

# ==========================================
# 3. DEFINE CHANNELS
# ==========================================
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    
]

# ==========================================
# 4. RUN UMAP GRID
# ==========================================
fig, axes = plt.subplots(2, 3, figsize=(28, 18))
axes = axes.flatten()

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    if not config['indices']:
        ax.set_title(f"{config['name']} (0 feats)")
        continue
        
    print(f"Running UMAP for {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    texts = []
    
    for plate in plate_list:
        mask = df['Plate'] == plate
        p_embed = embedding[mask]
        p_meta = df[mask]
        
        # Check for control (handling both possible naming conventions)
        is_ctrl = p_meta['Treatment'].isin(["no_sgRNA", "nosgrna"])
        color = plate_color_map[plate]
        
        # Plot Mutants
        ax.scatter(p_embed[~is_ctrl, 0], p_embed[~is_ctrl, 1], 
                   color=color, marker='o', s=80, alpha=0.6, label=f"{plate}")
        
        # Plot Controls
        ax.scatter(p_embed[is_ctrl, 0], p_embed[is_ctrl, 1], 
                   edgecolors=color, facecolors='none', 
                   marker='s', s=180, linewidths=3)

        # Add Labels
        for i, row in enumerate(p_meta.itertuples()):
            if row.Treatment not in ["no_sgRNA", "nosgrna"]:
                label = f"{row.Treatment}_{row.Plate.split('_')[-1]}"
                texts.append(ax.text(p_embed[i, 0], p_embed[i, 1], label, 
                                     fontsize=8, color=color, fontweight='bold'))

    ax.set_title(f"{config['name']} ({len(config['indices'])} feats)", fontsize=18, fontweight='bold')
    if idx == 0:
        ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10, title="Plate & Timepoint")
    
    adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='gray', alpha=0.3))

plt.suptitle("CRISPRi UMAP Analysis (Filtered Plates)", fontsize=28, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles.csv")
df = pd.read_csv(file_path)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x in ["no_sgRNA", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'UMAP1': False, 'UMAP2': False},
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) # Tiny mutants
    fig.update_traces(marker=dict(size=8, line=dict(width=1)), selector=dict(marker_symbol='square')) # Clear controls
    
    fig.update_layout(
        width=1000, 
        height=700,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Show the plot (In Jupyter, these will appear one after another)
    fig.show()
    
    # OPTIONAL: Save to HTML so you can send them to others or open later
    # fig.write_html(os.path.join(PROJECT_ROOT, f"umap_{config['name'].replace(' ', '_')}.html"))

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)
feature_cols = [str(i) for i in range(6400)]
df_raw.columns = [str(c) for c in df_raw.columns]

# ==========================================
# STEP 1: ACROSS-PLATE STABILITY
# Goal: Remove features that vary wildly between control wells of different plates.
# ==========================================
def filter_step1_across_plate_stability(df, features, top_n_to_keep=2000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    variance_filter = df[features].std() > 0.01
    stable_features = total_batch_noise[variance_filter].sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

print("\nRunning Step 1: Across-Plate Stability...")
stable_cols = filter_step1_across_plate_stability(df_raw, feature_cols, top_n_to_keep=2000)
df_step1 = df_raw[['Plate', 'Well_ID', 'Treatment'] + stable_cols]
print(f"Step 1 Complete: Kept {len(stable_cols)} features.")

# ==========================================
# STEP 2: T0 NEUTRALITY
# Goal: Remove features that 'hallucinate' differences between mutants and controls at T0.
# ==========================================
def filter_step2_t0_neutrality(df, features, top_n_to_keep=1000):
    t0_df = df[df['Plate'].str.endswith('T0')].copy()
    t0_ctrl_median = t0_df[t0_df['Treatment'] == CONTROL_LABEL][features].median()
    t0_mutants = t0_df[t0_df['Treatment'] != CONTROL_LABEL]
    
    t0_bias_score = (t0_mutants[features] - t0_ctrl_median).abs().mean()
    stable_at_start = t0_bias_score.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_at_start

print("\nRunning Step 2: T0 Neutrality...")
neutral_features = filter_step2_t0_neutrality(df_step1, stable_cols, top_n_to_keep=1000)
df_step2 = df_step1[['Plate', 'Well_ID', 'Treatment'] + neutral_features]
print(f"Step 2 Complete: Kept {len(neutral_features)} features.")

# ==========================================
# STEP 3: TEMPORAL STABILITY (T0 vs T1 Drift)
# Goal: Remove features that drift in the controls between T0 and T1.
# ==========================================
def filter_step3_temporal_stability(df, features, top_n_to_keep=500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()
    base_plate_names = set([p.split('_')[0] for p in plate_medians.index])
    
    temporal_drifts = []
    for base in base_plate_names:
        p0, p1 = f"{base}_T0", f"{base}_T1"
        if p0 in plate_medians.index and p1 in plate_medians.index:
            drift = (plate_medians.loc[p1] - plate_medians.loc[p0]).abs()
            temporal_drifts.append(drift)
    
    avg_temporal_drift = pd.concat(temporal_drifts, axis=1).mean(axis=1)
    stable_over_time = avg_temporal_drift.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_over_time

print("\nRunning Step 3: Temporal Stability...")
stable_time_features = filter_step3_temporal_stability(df_step2, neutral_features, top_n_to_keep=500)
df_step3 = df_step2[['Plate', 'Well_ID', 'Treatment'] + stable_time_features]
print(f"Step 3 Complete: Kept {len(stable_time_features)} features.")

# ==========================================
# STEP 4: AGGRESSIVE PLATE CONVERGENCE
# Goal: Final prune of features causing the most disagreement between plates.
# ==========================================
def filter_step4_plate_convergence(df, features, top_n_to_keep=200):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_noises.append(noise)
    
    avg_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)
    plate_blind_features = avg_plate_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return plate_blind_features

print("\nRunning Step 4: Aggressive Plate Convergence...")
final_vetted_features = filter_step4_plate_convergence(df_step3, stable_time_features, top_n_to_keep=200)
df_step4 = df_step3[['Plate', 'Well_ID', 'Treatment'] + final_vetted_features]
print(f"Step 4 Complete: Kept {len(final_vetted_features)} features.")

# ==========================================
# STEP 5: REDUNDANCY REMOVAL
# Goal: Remove highly correlated features to balance phenotypic signals.
# ==========================================
def filter_step5_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

print("\nRunning Step 5: Redundancy Filter...")
non_redundant_features = filter_step5_redundancy(df_step4, final_vetted_features, correlation_threshold=0.9)
df_final_vetted = df_step4[['Plate', 'Well_ID', 'Treatment'] + non_redundant_features]

# ==========================================
# FINAL SAVE
# ==========================================
output_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles2.csv")
df_final_vetted.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Final feature count: {len(non_redundant_features)}")
print(f"Master profiles saved to: {output_path}")
print("="*40)

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles2.csv")
df = pd.read_csv(file_path)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x in ["no_sgRNA", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'UMAP1': False, 'UMAP2': False},
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) # Tiny mutants
    fig.update_traces(marker=dict(size=8, line=dict(width=1)), selector=dict(marker_symbol='square')) # Clear controls
    
    fig.update_layout(
        width=1000, 
        height=700,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Show the plot (In Jupyter, these will appear one after another)
    fig.show()
    
    # OPTIONAL: Save to HTML so you can send them to others or open later
    # fig.write_html(os.path.join(PROJECT_ROOT, f"umap_{config['name'].replace(' ', '_')}.html"))

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go

# ==========================================
# 1. LOAD AND MERGE DATA
# ==========================================
PATH_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"

# Load Master Profiles
df = pd.read_csv(os.path.join(PATH_ROOT, "vetted_final_master_profiles.csv"))
features = [c for c in df.columns if c.isdigit()]

# Load Annotation Excel (Update extension if it's .xlsx or .xls)
# Assuming 'Treatment' in df matches a column in the Excel (e.g., 'Gene' or 'Treatment')
anno_df = pd.read_excel(os.path.join(PATH_ROOT, "Pathway_annotation.xlsx"))

# Merge metadata
# Adjust 'left_on' if the gene name column in your Excel is called something else
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3']], on='Treatment', how='left')

# ==========================================
# 2. PRE-PROCESSING FOR PLOT
# ==========================================
# Extract Timepoint from Plate string (e.g., "PLATE1_T0" -> "T0")
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

# Handle Timepoint 0 Logic: Set category to "Baseline" for T0
df['Plot_Category'] = df['SubtiWiki Annotation 3']
df.loc[df['Timepoint'] == 'T0', 'Plot_Category'] = 'Baseline (T0)'

# Define Symbol Map: T1=cross, T2=circle, T0=circle (or as preferred)
symbol_map = {"T0": "circle", "T1": "x", "T2": "circle"}

# ==========================================
# 3. RUN UMAP
# ==========================================
X_scaled = StandardScaler().fit_transform(df[features].values)
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
embedding = reducer.fit_transform(X_scaled)

df['UMAP1'] = embedding[:, 0]
df['UMAP2'] = embedding[:, 1]

# ==========================================
# 4. CREATE INTERACTIVE OVERLAY PLOT
# ==========================================
# Sort so T0 is plotted first (in the background)
df = df.sort_values('Timepoint')

fig = px.scatter(
    df,
    x='UMAP1',
    y='UMAP2',
    color='Plot_Category',
    symbol='Timepoint',
    symbol_map=symbol_map,
    hover_name='Treatment',
    hover_data=['Plate', 'SubtiWiki Annotation 3'],
    title="UMAP: Functional Pathway Overlay (SubtiWiki Annotation 3)",
    template='plotly_white',
    color_discrete_map={'Baseline (T0)': 'lightgrey'} # Force T0 to be grey
)

# Refine marker sizes
fig.update_traces(marker=dict(size=6, opacity=0.8))

# Final layout tweaks
fig.update_layout(
    width=1200,
    height=800,
    legend_title_text='Pathway / Timepoint',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
)

fig.show()

# Save as HTML for your records
# fig.write_html(os.path.join(PATH_ROOT, "UMAP_pathway_overlay.html"))

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go

# ==========================================
# 1. LOAD AND MERGE DATA
# ==========================================
PATH_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"

# Load Master Profiles
df = pd.read_csv(os.path.join(PATH_ROOT, "vetted_final_master_profiles.csv"))
features = [c for c in df.columns if c.isdigit()]

# Load Annotation Excel (Update extension if it's .xlsx or .xls)
# Assuming 'Treatment' in df matches a column in the Excel (e.g., 'Gene' or 'Treatment')
anno_df = pd.read_excel(os.path.join(PATH_ROOT, "Pathway_annotation.xlsx"))

# Merge metadata
# Adjust 'left_on' if the gene name column in your Excel is called something else
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 4']], on='Treatment', how='left')

# ==========================================
# 2. PRE-PROCESSING FOR PLOT
# ==========================================
# Extract Timepoint from Plate string (e.g., "PLATE1_T0" -> "T0")
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

# Handle Timepoint 0 Logic: Set category to "Baseline" for T0
df['Plot_Category'] = df['SubtiWiki Annotation 4']
df.loc[df['Timepoint'] == 'T0', 'Plot_Category'] = 'Baseline (T0)'

# Define Symbol Map: T1=cross, T2=circle, T0=circle (or as preferred)
symbol_map = {"T0": "circle", "T1": "x", "T2": "circle"}

# ==========================================
# 3. RUN UMAP
# ==========================================
X_scaled = StandardScaler().fit_transform(df[features].values)
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
embedding = reducer.fit_transform(X_scaled)

df['UMAP1'] = embedding[:, 0]
df['UMAP2'] = embedding[:, 1]

# ==========================================
# 4. CREATE INTERACTIVE OVERLAY PLOT
# ==========================================
# Sort so T0 is plotted first (in the background)
df = df.sort_values('Timepoint')

fig = px.scatter(
    df,
    x='UMAP1',
    y='UMAP2',
    color='Plot_Category',
    symbol='Timepoint',
    symbol_map=symbol_map,
    hover_name='Treatment',
    hover_data=['Plate', 'SubtiWiki Annotation 4'],
    title="UMAP: Functional Pathway Overlay (SubtiWiki Annotation 4)",
    template='plotly_white',
    color_discrete_map={'Baseline (T0)': 'lightgrey'} # Force T0 to be grey
)

# Refine marker sizes
fig.update_traces(marker=dict(size=6, opacity=0.8))

# Final layout tweaks
fig.update_layout(
    width=1200,
    height=800,
    legend_title_text='Pathway / Timepoint',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
)

fig.show()

# Save as HTML for your records
# fig.write_html(os.path.join(PATH_ROOT, "UMAP_pathway_overlay.html"))

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# ==========================================
# 1. SETUP AND DATA LOADING
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

df = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# Merge with SubtiWiki Annotation
# Note: 'Treatment' in your CSV must match 'Treatment' in the Excel
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 4']], on='Treatment', how='left')

# Fill missing annotations with "Unknown/Other" to color them black later
df['SubtiWiki Annotation 3'] = df['SubtiWiki Annotation 4'].fillna("Unknown/Other")

# Extract Timepoint for shapes and grey-out logic
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

# Create a Display Category: T0 is "Baseline", others use SubtiWiki
df['Display_Category'] = df['SubtiWiki Annotation 4']
df.loc[df['Timepoint'] == 'T0', 'Display_Category'] = 'Baseline (T0)'

# ==========================================
# 2. DEFINE CHANNEL GROUPS
# ==========================================
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# ==========================================
# 3. COLOR & SYMBOL CONFIG
# ==========================================
# Force T0 to be Grey and Unknowns to be Black
color_map = {
    'Baseline (T0)': 'lightgrey',
    'Unknown/Other': 'black'
}

# Map symbols to timepoints
symbol_map = {"T0": "circle", "T1": "x", "T2": "circle"}

# ==========================================
# 4. RUN UMAP LOOP FOR ALL CHANNELS
# ==========================================
for config in plot_configs:
    if not config['indices']:
        continue
    
    print(f"Generating Pathway UMAP for: {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create Figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Display_Category',
        symbol='Timepoint',
        symbol_map=symbol_map,
        hover_name='Treatment',
        hover_data={'Plate': True, 'SubtiWiki Annotation 3': True, 'UMAP1': False, 'UMAP2': False},
        title=f"Pathway Overlay: {config['name']} ({len(config['indices'])} features)",
        color_discrete_map=color_map, # Applies our grey/black rules
        template='plotly_white'
    )
    
    # Adjust marker sizes
    fig.update_traces(marker=dict(size=5, opacity=0.8), selector=dict(marker_symbol='circle'))
    fig.update_traces(marker=dict(size=6), selector=dict(marker_symbol='x'))
    
    # Layout Tweeks
    fig.update_layout(
        width=1100,
        height=700,
        legend_title_text='Pathway / Timepoint',
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
    )
    
    fig.show()

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# --- 1. SETUP ---
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "UMAP_Interactive_Plots")

# Create the output directory if it doesn't exist
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

df = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df.columns = [str(c) for c in df.columns]

# --- 2. MERGE & FIX ANNOTATIONS ---
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
df['SubtiWiki Annotation 4'] = df['SubtiWiki Annotation 4'].fillna("Unknown/Other")
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

df['Display_Category'] = df['SubtiWiki Annotation 4']
df.loc[df['Timepoint'] == 'T0', 'Display_Category'] = 'Baseline (T0)'

# --- 3. DYNAMIC COLOR MAPPING ---
all_cats = df['Display_Category'].unique()
standard_colors = px.colors.qualitative.Alphabet 

color_map = {cat: standard_colors[i % len(standard_colors)] for i, cat in enumerate(all_cats)}
color_map['Baseline (T0)'] = 'lightgrey'
color_map['Unknown/Other'] = 'black'

# --- 4. RUN UMAP LOOP AND SAVE ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Vetted_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Generating and Saving UMAP for: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    fig = px.scatter(
        df, x='UMAP1', y='UMAP2', 
        color='Display_Category',
        symbol='Timepoint',
        symbol_map={"T0": "circle", "T1": "x", "T2": "circle"},
        hover_name='Treatment',
        hover_data=['Plate', 'SubtiWiki Annotation 4'],
        title=f"Pathway Overlay: {config['name'].replace('_', ' ')}",
        color_discrete_map=color_map,
        template='plotly_white'
    )
    
    fig.update_traces(marker=dict(opacity=1.0)) 
    fig.update_traces(marker=dict(size=5), selector=dict(marker_symbol='circle'))
    fig.update_traces(marker=dict(size=6), selector=dict(marker_symbol='x'))
    
    fig.update_layout(width=1100, height=700)
    
    # --- SAVE TO LOCATION ---
    save_path = os.path.join(OUTPUT_DIR, f"UMAP_{config['name']}.html")
    fig.write_html(save_path)
    
    # Optional: also show in the notebook
    fig.show()

print(f"\nAll interactive plots have been saved to: {OUTPUT_DIR}")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
# Create a dedicated folder for the HTML plots
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "UMAP_HTML_Reports")
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles2.csv")
df = pd.read_csv(file_path)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x in ["no_sgRNA", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing and saving interactive UMAP for: {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'UMAP1': False, 'UMAP2': False},
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=8, line=dict(width=1)), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=1000, 
        height=700,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Show the plot
    fig.show()
    
    # --- SAVE TO HTML ---
    # We replace spaces with underscores for the filename
    safe_name = config['name'].replace(' ', '_')
    save_path = os.path.join(OUTPUT_DIR, f"umap_{safe_name}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")

print(f"\nFinished! All interactive plots are available in: {OUTPUT_DIR}")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "UMAP_MultiShade_Plots")
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles2.csv")
df = pd.read_csv(file_path)
df.columns = [str(c) for c in df.columns]

# 2. DEFINE THE NESTED COLOR MAP
# We manually define shades for each Plate/Timepoint combination
nested_color_map = {
    # T0 - The Blue Family (Baseline)
    "PLATE1_T0": "#DEEBF7", "PLATE2_T0": "#C6DBEF", "PLATE3_T0": "#9ECAE1", 
    "PLATE4_T0": "#6BAED6", "PLATE5_T0": "#4292C6",
    
    # T1 - The Orange/Red Family (Early Response)
    "PLATE1_T1": "#FEE6CE", "PLATE2_T1": "#FDD0A2", "PLATE3_T1": "#FDAE6B", 
    "PLATE4_T1": "#FD8D3C", "PLATE5_T1": "#F16913",
    
    # T2 - The Purple/Dark Family (Final Phenotype)
    "PLATE1_T2": "#EFEDF5", "PLATE2_T2": "#DADAEB", "PLATE3_T2": "#BCBDDC", 
    "PLATE4_T2": "#9E9AC8", "PLATE5_T2": "#807DBA"
}

# Create a 'Type' column for shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x in ["no_sgRNA", "nosgrna"] else 'Mutant')

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# 4. RUN UMAP LOOP
for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Processing Multi-Shade UMAP for: {config['name']}...")
    
    # Reduce
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df_plot = df.copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    # Sort for consistent layering (T0 at back, T2 at front)
    df_plot['Timepoint_Rank'] = df_plot['Plate'].str.extract(r'_(T\d)')[0]
    df_plot = df_plot.sort_values('Timepoint_Rank')

    fig = px.scatter(
        df_plot, 
        x='UMAP1', y='UMAP2', 
        color='Plate',
        color_discrete_map=nested_color_map, 
        symbol='Type',
        symbol_map={'Control': 'square', 'Mutant': 'circle'},
        hover_name='Treatment',
        hover_data=['Plate', 'Well_ID'],
        title=f"Multi-Shade Timepoint UMAP: {config['name']}",
        template='plotly_white'
    )
    
    # Visual Density Tweaks
    fig.update_traces(marker=dict(size=5, opacity=0.8), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=8, line=dict(width=1, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=1100, 
        height=800, 
        legend_title_text='Plate & Timepoint Gradient'
    )
    
    # SAVE & SHOW
    save_name = config['name'].replace(' ', '_')
    fig.write_html(os.path.join(OUTPUT_DIR, f"multishade_umap_{save_name}.html"))
    fig.show()

print(f"Success! Multi-shade plots saved to: {OUTPUT_DIR}")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# --- 1. SETUP ---
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "UMAP_Phenotype_Gradients")
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

master_file = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles.csv")
pheno_file = os.path.join(PROJECT_ROOT, "Phenotype.xlsx")

df = pd.read_csv(master_file)
pheno_df = pd.read_excel(pheno_file) 
df.columns = [str(c) for c in df.columns]

# --- 2. MERGE & CLEAN ---
# Merge based on the first column of the Excel (Treatment/Gene)
df = df.merge(pheno_df, left_on='Treatment', right_on=pheno_df.columns[0], how='left')
pheno_cols = pheno_df.columns[1:7] # The 6 phenotype columns

# Extract Timepoint for logic
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

# --- 3. FILTER FOR CHANNEL 1 ---
vetted_features = [c for c in df.columns if c.isdigit()]
channel_1_indices = [f for f in vetted_features if 0 <= int(f) < 1280]

# --- 4. RUN UMAP (Channel 1 Only) ---
print("Computing UMAP for Channel 1...")
X_scaled = StandardScaler().fit_transform(df[channel_1_indices].values)
embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)

df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]

# --- 5. GENERATE 6 PLOTS WITH T0 GREY-OUT ---
for pheno in pheno_cols:
    print(f"Generating Plot: {pheno}...")
    
    # Create a temporary column for this specific plot's colors
    plot_col = f"{pheno}_display"
    
    # Logic: If T0, set to -1 (which we will map to Grey). Otherwise, use 0, 1, 2.
    df[plot_col] = df[pheno].fillna(0).astype(float)
    df.loc[df['Timepoint'] == 'T0', plot_col] = -1.0 # Force T0 to special value
    
    # Define Custom Color Scale: -1=Grey, 0=LightBlue/White, 1=Yellow, 2=Red
    # We use a discrete map style with a continuous scale to handle the gradient
    fig = px.scatter(
        df, x='UMAP1', y='UMAP2',
        color=plot_col,
        # Scale: -1 (Grey) -> 0 (Soft Grey/White) -> 1 (Yellow) -> 2 (Red)
        color_continuous_scale=[
            (0.00, "#D3D3D3"), # -1.0: Light Grey
            (0.33, "#D3D3D3"), # Transition
            (0.34, "#F0F0F0"), # 0.0: Baseline White/Grey
            (0.66, "#F1C40F"), # 1.0: Intermediate Yellow
            (1.00, "#E74C3C")  # 2.0: Strong Red
        ],
        symbol='Timepoint',
        symbol_map={"T0": "circle", "T1": "x", "T2": "circle"},
        hover_name='Treatment',
        hover_data=['Plate', 'Well_ID', pheno],
        title=f"Phenotype Progression: {pheno} (T0 = Grey)",
        template='plotly_white'
    )
    
    # Marker styling
    fig.update_traces(marker=dict(size=6, opacity=0.9))
    
    # Adjust colorbar to hide the -1 value and show 0, 1, 2
    fig.update_coloraxes(
        colorbar=dict(
            tickvals=[0, 1, 2], 
            ticktext=['None (0)', 'Int. (1)', 'Strong (2)'],
            title="Intensity"
        ),
        cmin=-1, cmax=2
    )
    
    fig.update_layout(width=1100, height=750)
    
    # Save
    save_path = os.path.join(OUTPUT_DIR, f"Pheno_Gradient_{pheno.replace(' ', '_')}.html")
    fig.write_html(save_path)
    fig.show()

print(f"\nDone! Plots saved to: {OUTPUT_DIR}")

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)
feature_cols = [str(i) for i in range(6400)]
df_raw.columns = [str(c) for c in df_raw.columns]

# ==========================================
# STEP 1: ACROSS-PLATE STABILITY
# ==========================================
def filter_step1_across_plate_stability(df, features, top_n_to_keep=2000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    variance_filter = df[features].std() > 0.01
    stable_features = total_batch_noise[variance_filter].sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

print("\nRunning Step 1: Across-Plate Stability...")
stable_cols = filter_step1_across_plate_stability(df_raw, feature_cols, top_n_to_keep=2000)
df_step1 = df_raw[['Plate', 'Well_ID', 'Treatment'] + stable_cols]

# ==========================================
# STEP 2: T0 NEUTRALITY
# ==========================================
def filter_step2_t0_neutrality(df, features, top_n_to_keep=1000):
    t0_df = df[df['Plate'].str.endswith('T0')].copy()
    t0_ctrl_median = t0_df[t0_df['Treatment'] == CONTROL_LABEL][features].median()
    t0_mutants = t0_df[t0_df['Treatment'] != CONTROL_LABEL]
    
    t0_bias_score = (t0_mutants[features] - t0_ctrl_median).abs().mean()
    stable_at_start = t0_bias_score.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_at_start

print("Running Step 2: T0 Neutrality...")
neutral_features = filter_step2_t0_neutrality(df_step1, stable_cols, top_n_to_keep=1000)
df_step2 = df_step1[['Plate', 'Well_ID', 'Treatment'] + neutral_features]

# ==========================================
# STEP 3: TEMPORAL STABILITY
# ==========================================
def filter_step3_temporal_stability(df, features, top_n_to_keep=500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()
    base_plate_names = set([p.split('_')[0] for p in plate_medians.index])
    
    temporal_drifts = []
    for base in base_plate_names:
        p0, p1 = f"{base}_T0", f"{base}_T1"
        if p0 in plate_medians.index and p1 in plate_medians.index:
            drift = (plate_medians.loc[p1] - plate_medians.loc[p0]).abs()
            temporal_drifts.append(drift)
    
    avg_temporal_drift = pd.concat(temporal_drifts, axis=1).mean(axis=1)
    stable_over_time = avg_temporal_drift.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_over_time

print("Running Step 3: Temporal Stability...")
stable_time_features = filter_step3_temporal_stability(df_step2, neutral_features, top_n_to_keep=500)
df_step3 = df_step2[['Plate', 'Well_ID', 'Treatment'] + stable_time_features]

# ==========================================
# STEP 4: AGGRESSIVE PLATE CONVERGENCE
# ==========================================
def filter_step4_plate_convergence(df, features, top_n_to_keep=200):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_noises.append(noise)
    
    avg_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)
    plate_blind_features = avg_plate_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return plate_blind_features

print("Running Step 4: Aggressive Plate Convergence...")
final_vetted_features = filter_step4_plate_convergence(df_step3, stable_time_features, top_n_to_keep=200)

# ==========================================
# STEP 4.5: WITHIN-PLATE CONSISTENCY
# ==========================================
def filter_within_plate_consistency(df, features, top_n_to_keep=200):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    # Standard deviation of identical control wells within each plate
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

print("Running Step 4.5: Within-Plate Consistency...")
consistent_features = filter_within_plate_consistency(df_step3, final_vetted_features, top_n_to_keep=200)
df_step4 = df_step3[['Plate', 'Well_ID', 'Treatment'] + consistent_features]

# ==========================================
# STEP 5: REDUNDANCY REMOVAL
# ==========================================
def filter_step5_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

print("Running Step 5: Redundancy Filter...")
non_redundant_features = filter_step5_redundancy(df_step4, consistent_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final_vetted = df_step4[['Plate', 'Well_ID', 'Treatment'] + non_redundant_features]
output_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles3.csv")
df_final_vetted.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Final feature count: {len(non_redundant_features)}")
print(f"Master profiles saved to: {output_path}")
print("="*40)

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles3.csv")
df = pd.read_csv(file_path)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x in ["no_sgRNA", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'UMAP1': False, 'UMAP2': False},
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) # Tiny mutants
    fig.update_traces(marker=dict(size=8, line=dict(width=1)), selector=dict(marker_symbol='square')) # Clear controls
    
    fig.update_layout(
        width=1000, 
        height=700,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Show the plot (In Jupyter, these will appear one after another)
    fig.show()
    
    # OPTIONAL: Save to HTML so you can send them to others or open later
    fig.write_html(os.path.join(PROJECT_ROOT, f"whithincorrumap_{config['name'].replace(' ', '_')}.html"))

In [ ]:
#dit is ook wel een goeie

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)
feature_cols = [str(i) for i in range(6400)]
df_raw.columns = [str(c) for c in df_raw.columns]

# ==========================================
# STEP 1: ACROSS-PLATE STABILITY
# ==========================================
def filter_step1_across_plate_stability(df, features, top_n_to_keep=800):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    variance_filter = df[features].std() > 0.01
    stable_features = total_batch_noise[variance_filter].sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

print("\nRunning Step 1: Across-Plate Stability...")
stable_cols = filter_step1_across_plate_stability(df_raw, feature_cols, top_n_to_keep=800)
df_step1 = df_raw[['Plate', 'Well_ID', 'Treatment'] + stable_cols]

# ==========================================
# STEP 2: T0 NEUTRALITY
# ==========================================
def filter_step2_t0_neutrality(df, features, top_n_to_keep=200):
    t0_df = df[df['Plate'].str.endswith('T0')].copy()
    t0_ctrl_median = t0_df[t0_df['Treatment'] == CONTROL_LABEL][features].median()
    t0_mutants = t0_df[t0_df['Treatment'] != CONTROL_LABEL]
    
    t0_bias_score = (t0_mutants[features] - t0_ctrl_median).abs().mean()
    stable_at_start = t0_bias_score.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_at_start

print("Running Step 2: T0 Neutrality...")
neutral_features = filter_step2_t0_neutrality(df_step1, stable_cols, top_n_to_keep=200)
df_step2 = df_step1[['Plate', 'Well_ID', 'Treatment'] + neutral_features]

# ==========================================
# STEP 3: TEMPORAL STABILITY
# ==========================================
def filter_step3_temporal_stability(df, features, top_n_to_keep=200):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()
    base_plate_names = set([p.split('_')[0] for p in plate_medians.index])
    
    temporal_drifts = []
    for base in base_plate_names:
        p0, p1 = f"{base}_T0", f"{base}_T1"
        if p0 in plate_medians.index and p1 in plate_medians.index:
            drift = (plate_medians.loc[p1] - plate_medians.loc[p0]).abs()
            temporal_drifts.append(drift)
    
    avg_temporal_drift = pd.concat(temporal_drifts, axis=1).mean(axis=1)
    stable_over_time = avg_temporal_drift.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_over_time

print("Running Step 3: Temporal Stability...")
stable_time_features = filter_step3_temporal_stability(df_step2, neutral_features, top_n_to_keep=200)
df_step3 = df_step2[['Plate', 'Well_ID', 'Treatment'] + stable_time_features]

# ==========================================
# STEP 4: AGGRESSIVE PLATE CONVERGENCE
# ==========================================
def filter_step4_plate_convergence(df, features, top_n_to_keep=200):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_noises.append(noise)
    
    avg_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)
    plate_blind_features = avg_plate_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return plate_blind_features

print("Running Step 4: Aggressive Plate Convergence...")
final_vetted_features = filter_step4_plate_convergence(df_step3, stable_time_features, top_n_to_keep=200)

# ==========================================
# STEP 4.5: WITHIN-PLATE CONSISTENCY
# ==========================================
def filter_within_plate_consistency(df, features, top_n_to_keep=500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    # Standard deviation of identical control wells within each plate
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

print("Running Step 4.5: Within-Plate Consistency...")
consistent_features = filter_within_plate_consistency(df_step3, final_vetted_features, top_n_to_keep=500)
df_step4 = df_step3[['Plate', 'Well_ID', 'Treatment'] + consistent_features]

# ==========================================
# STEP 5: REDUNDANCY REMOVAL
# ==========================================
def filter_step5_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

print("Running Step 5: Redundancy Filter...")
non_redundant_features = filter_step5_redundancy(df_step4, consistent_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final_vetted = df_step4[['Plate', 'Well_ID', 'Treatment'] + non_redundant_features]
output_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles3.csv")
df_final_vetted.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Final feature count: {len(non_redundant_features)}")
print(f"Master profiles saved to: {output_path}")
print("="*40)

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles3.csv")
df = pd.read_csv(file_path)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x in ["no_sgRNA", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'UMAP1': False, 'UMAP2': False},
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) # Tiny mutants
    fig.update_traces(marker=dict(size=8, line=dict(width=1)), selector=dict(marker_symbol='square')) # Clear controls
    
    fig.update_layout(
        width=1000, 
        height=700,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Show the plot (In Jupyter, these will appear one after another)
    fig.show()
    
    # OPTIONAL: Save to HTML so you can send them to others or open later
    fig.write_html(os.path.join(PROJECT_ROOT, f"whithincorrumap_{config['name'].replace(' ', '_')}.html"))

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)
feature_cols = [str(i) for i in range(6400)]
df_raw.columns = [str(c) for c in df_raw.columns]

# ==========================================
# STEP 1: ACROSS-PLATE STABILITY
# ==========================================
def filter_step1_across_plate_stability(df, features, top_n_to_keep=500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    variance_filter = df[features].std() > 0.01
    stable_features = total_batch_noise[variance_filter].sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

print("\nRunning Step 1: Across-Plate Stability...")
stable_cols = filter_step1_across_plate_stability(df_raw, feature_cols, top_n_to_keep=500)
df_step1 = df_raw[['Plate', 'Well_ID', 'Treatment'] + stable_cols]

# ==========================================
# STEP 2: T0 NEUTRALITY
# ==========================================
def filter_step2_t0_neutrality(df, features, top_n_to_keep=100):
    t0_df = df[df['Plate'].str.endswith('T0')].copy()
    t0_ctrl_median = t0_df[t0_df['Treatment'] == CONTROL_LABEL][features].median()
    t0_mutants = t0_df[t0_df['Treatment'] != CONTROL_LABEL]
    
    t0_bias_score = (t0_mutants[features] - t0_ctrl_median).abs().mean()
    stable_at_start = t0_bias_score.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_at_start

print("Running Step 2: T0 Neutrality...")
neutral_features = filter_step2_t0_neutrality(df_step1, stable_cols, top_n_to_keep=100)
df_step2 = df_step1[['Plate', 'Well_ID', 'Treatment'] + neutral_features]

# ==========================================
# STEP 3: TEMPORAL STABILITY
# ==========================================
def filter_step3_temporal_stability(df, features, top_n_to_keep=100):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()
    base_plate_names = set([p.split('_')[0] for p in plate_medians.index])
    
    temporal_drifts = []
    for base in base_plate_names:
        p0, p1 = f"{base}_T0", f"{base}_T1"
        if p0 in plate_medians.index and p1 in plate_medians.index:
            drift = (plate_medians.loc[p1] - plate_medians.loc[p0]).abs()
            temporal_drifts.append(drift)
    
    avg_temporal_drift = pd.concat(temporal_drifts, axis=1).mean(axis=1)
    stable_over_time = avg_temporal_drift.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_over_time

print("Running Step 3: Temporal Stability...")
stable_time_features = filter_step3_temporal_stability(df_step2, neutral_features, top_n_to_keep=100)
df_step3 = df_step2[['Plate', 'Well_ID', 'Treatment'] + stable_time_features]

# ==========================================
# STEP 4: AGGRESSIVE PLATE CONVERGENCE
# ==========================================
def filter_step4_plate_convergence(df, features, top_n_to_keep=100):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_noises.append(noise)
    
    avg_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)
    plate_blind_features = avg_plate_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return plate_blind_features

print("Running Step 4: Aggressive Plate Convergence...")
final_vetted_features = filter_step4_plate_convergence(df_step3, stable_time_features, top_n_to_keep=100)

# ==========================================
# STEP 4.5: WITHIN-PLATE CONSISTENCY
# ==========================================
def filter_within_plate_consistency(df, features, top_n_to_keep=100):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    # Standard deviation of identical control wells within each plate
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

print("Running Step 4.5: Within-Plate Consistency...")
consistent_features = filter_within_plate_consistency(df_step3, final_vetted_features, top_n_to_keep=100)
df_step4 = df_step3[['Plate', 'Well_ID', 'Treatment'] + consistent_features]

# ==========================================
# STEP 5: REDUNDANCY REMOVAL
# ==========================================
def filter_step5_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

print("Running Step 5: Redundancy Filter...")
non_redundant_features = filter_step5_redundancy(df_step4, consistent_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final_vetted = df_step4[['Plate', 'Well_ID', 'Treatment'] + non_redundant_features]
output_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles6.csv")
df_final_vetted.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Final feature count: {len(non_redundant_features)}")
print(f"Master profiles saved to: {output_path}")
print("="*40)

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles6.csv")
df = pd.read_csv(file_path)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x in ["no_sgRNA", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'UMAP1': False, 'UMAP2': False},
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) # Tiny mutants
    fig.update_traces(marker=dict(size=8, line=dict(width=1)), selector=dict(marker_symbol='square')) # Clear controls
    
    fig.update_layout(
        width=1000, 
        height=700,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Show the plot (In Jupyter, these will appear one after another)
    fig.show()
    
    # OPTIONAL: Save to HTML so you can send them to others or open later
    fig.write_html(os.path.join(PROJECT_ROOT, f"whithincorrumap6_{config['name'].replace(' ', '_')}.html"))

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)
feature_cols = [str(i) for i in range(6400)]
df_raw.columns = [str(c) for c in df_raw.columns]

# ==========================================
# STEP 1: ACROSS-PLATE STABILITY
# ==========================================
def filter_step1_across_plate_stability(df, features, top_n_to_keep=1000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    variance_filter = df[features].std() > 0.01
    stable_features = total_batch_noise[variance_filter].sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

print("\nRunning Step 1: Across-Plate Stability...")
stable_cols = filter_step1_across_plate_stability(df_raw, feature_cols, top_n_to_keep=1000)
df_step1 = df_raw[['Plate', 'Well_ID', 'Treatment'] + stable_cols]

# ==========================================
# STEP 2: T0 NEUTRALITY
# ==========================================
def filter_step2_t0_neutrality(df, features, top_n_to_keep=500):
    t0_df = df[df['Plate'].str.endswith('T0')].copy()
    t0_ctrl_median = t0_df[t0_df['Treatment'] == CONTROL_LABEL][features].median()
    t0_mutants = t0_df[t0_df['Treatment'] != CONTROL_LABEL]
    
    t0_bias_score = (t0_mutants[features] - t0_ctrl_median).abs().mean()
    stable_at_start = t0_bias_score.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_at_start

print("Running Step 2: T0 Neutrality...")
neutral_features = filter_step2_t0_neutrality(df_step1, stable_cols, top_n_to_keep=500)
df_step2 = df_step1[['Plate', 'Well_ID', 'Treatment'] + neutral_features]

# ==========================================
# STEP 3: TEMPORAL STABILITY
# ==========================================
def filter_step3_temporal_stability(df, features, top_n_to_keep=500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()
    base_plate_names = set([p.split('_')[0] for p in plate_medians.index])
    
    temporal_drifts = []
    for base in base_plate_names:
        p0, p1 = f"{base}_T0", f"{base}_T1"
        if p0 in plate_medians.index and p1 in plate_medians.index:
            drift = (plate_medians.loc[p1] - plate_medians.loc[p0]).abs()
            temporal_drifts.append(drift)
    
    avg_temporal_drift = pd.concat(temporal_drifts, axis=1).mean(axis=1)
    stable_over_time = avg_temporal_drift.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_over_time

print("Running Step 3: Temporal Stability...")
stable_time_features = filter_step3_temporal_stability(df_step2, neutral_features, top_n_to_keep=500)
df_step3 = df_step2[['Plate', 'Well_ID', 'Treatment'] + stable_time_features]

# ==========================================
# STEP 4: AGGRESSIVE PLATE CONVERGENCE
# ==========================================
def filter_step4_plate_convergence(df, features, top_n_to_keep=500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_noises.append(noise)
    
    avg_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)
    plate_blind_features = avg_plate_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return plate_blind_features

print("Running Step 4: Aggressive Plate Convergence...")
final_vetted_features = filter_step4_plate_convergence(df_step3, stable_time_features, top_n_to_keep=500)

# ==========================================
# STEP 4.5: WITHIN-PLATE CONSISTENCY
# ==========================================
def filter_within_plate_consistency(df, features, top_n_to_keep=500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    # Standard deviation of identical control wells within each plate
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

print("Running Step 4.5: Within-Plate Consistency...")
consistent_features = filter_within_plate_consistency(df_step3, final_vetted_features, top_n_to_keep=500)
df_step4 = df_step3[['Plate', 'Well_ID', 'Treatment'] + consistent_features]

# ==========================================
# STEP 5: REDUNDANCY REMOVAL
# ==========================================
def filter_step5_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

print("Running Step 5: Redundancy Filter...")
non_redundant_features = filter_step5_redundancy(df_step4, consistent_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final_vetted = df_step4[['Plate', 'Well_ID', 'Treatment'] + non_redundant_features]
output_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles7.csv")
df_final_vetted.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Final feature count: {len(non_redundant_features)}")
print(f"Master profiles saved to: {output_path}")
print("="*40)

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles7.csv")
df = pd.read_csv(file_path)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x in ["no_sgRNA", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'UMAP1': False, 'UMAP2': False},
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) # Tiny mutants
    fig.update_traces(marker=dict(size=8, line=dict(width=1)), selector=dict(marker_symbol='square')) # Clear controls
    
    fig.update_layout(
        width=1000, 
        height=700,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Show the plot (In Jupyter, these will appear one after another)
    fig.show()
    
    # OPTIONAL: Save to HTML so you can send them to others or open later
    fig.write_html(os.path.join(PROJECT_ROOT, f"whithincorrumap7_{config['name'].replace(' ', '_')}.html"))

In [ ]:
#ookvrijgoed

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)
feature_cols = [str(i) for i in range(6400)]
df_raw.columns = [str(c) for c in df_raw.columns]

# ==========================================
# STEP 1: ACROSS-PLATE STABILITY
# ==========================================
def filter_step1_across_plate_stability(df, features, top_n_to_keep=1000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    variance_filter = df[features].std() > 0.01
    stable_features = total_batch_noise[variance_filter].sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

print("\nRunning Step 1: Across-Plate Stability...")
stable_cols = filter_step1_across_plate_stability(df_raw, feature_cols, top_n_to_keep=1000)
df_step1 = df_raw[['Plate', 'Well_ID', 'Treatment'] + stable_cols]

# ==========================================
# STEP 2: T0 NEUTRALITY
# ==========================================
def filter_step2_t0_neutrality(df, features, top_n_to_keep=500):
    t0_df = df[df['Plate'].str.endswith('T0')].copy()
    t0_ctrl_median = t0_df[t0_df['Treatment'] == CONTROL_LABEL][features].median()
    t0_mutants = t0_df[t0_df['Treatment'] != CONTROL_LABEL]
    
    t0_bias_score = (t0_mutants[features] - t0_ctrl_median).abs().mean()
    stable_at_start = t0_bias_score.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_at_start

print("Running Step 2: T0 Neutrality...")
neutral_features = filter_step2_t0_neutrality(df_step1, stable_cols, top_n_to_keep=500)
df_step2 = df_step1[['Plate', 'Well_ID', 'Treatment'] + neutral_features]

# ==========================================
# STEP 3: TEMPORAL STABILITY
# ==========================================
def filter_step3_temporal_stability(df, features, top_n_to_keep=500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()
    base_plate_names = set([p.split('_')[0] for p in plate_medians.index])
    
    temporal_drifts = []
    for base in base_plate_names:
        p0, p1 = f"{base}_T0", f"{base}_T1"
        if p0 in plate_medians.index and p1 in plate_medians.index:
            drift = (plate_medians.loc[p1] - plate_medians.loc[p0]).abs()
            temporal_drifts.append(drift)
    
    avg_temporal_drift = pd.concat(temporal_drifts, axis=1).mean(axis=1)
    stable_over_time = avg_temporal_drift.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_over_time

print("Running Step 3: Temporal Stability...")
stable_time_features = filter_step3_temporal_stability(df_step2, neutral_features, top_n_to_keep=500)
df_step3 = df_step2[['Plate', 'Well_ID', 'Treatment'] + stable_time_features]

# ==========================================
# STEP 4: AGGRESSIVE PLATE CONVERGENCE
# ==========================================
def filter_step4_plate_convergence(df, features, top_n_to_keep=500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_noises.append(noise)
    
    avg_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)
    plate_blind_features = avg_plate_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return plate_blind_features

print("Running Step 4: Aggressive Plate Convergence...")
final_vetted_features = filter_step4_plate_convergence(df_step3, stable_time_features, top_n_to_keep=500)

# ==========================================
# STEP 4.5: WITHIN-PLATE CONSISTENCY
# ==========================================
def filter_within_plate_consistency(df, features, top_n_to_keep=300):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    # Standard deviation of identical control wells within each plate
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

print("Running Step 4.5: Within-Plate Consistency...")
consistent_features = filter_within_plate_consistency(df_step3, final_vetted_features, top_n_to_keep=300)
df_step4 = df_step3[['Plate', 'Well_ID', 'Treatment'] + consistent_features]

# ==========================================
# STEP 5: REDUNDANCY REMOVAL
# ==========================================
def filter_step5_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

print("Running Step 5: Redundancy Filter...")
non_redundant_features = filter_step5_redundancy(df_step4, consistent_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final_vetted = df_step4[['Plate', 'Well_ID', 'Treatment'] + non_redundant_features]
output_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles8.csv")
df_final_vetted.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Final feature count: {len(non_redundant_features)}")
print(f"Master profiles saved to: {output_path}")
print("="*40)

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles8.csv")
df = pd.read_csv(file_path)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x in ["no_sgRNA", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'UMAP1': False, 'UMAP2': False},
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) # Tiny mutants
    fig.update_traces(marker=dict(size=8, line=dict(width=1)), selector=dict(marker_symbol='square')) # Clear controls
    
    fig.update_layout(
        width=1000, 
        height=700,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Show the plot (In Jupyter, these will appear one after another)
    fig.show()
    
    # OPTIONAL: Save to HTML so you can send them to others or open later
    fig.write_html(os.path.join(PROJECT_ROOT, f"whithincorrumap8_{config['name'].replace(' ', '_')}.html"))

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)
feature_cols = [str(i) for i in range(6400)]
df_raw.columns = [str(c) for c in df_raw.columns]

# ==========================================
# STEP 1: ACROSS-PLATE STABILITY
# ==========================================
def filter_step1_across_plate_stability(df, features, top_n_to_keep=1500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    variance_filter = df[features].std() > 0.01
    stable_features = total_batch_noise[variance_filter].sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

print("\nRunning Step 1: Across-Plate Stability...")
stable_cols = filter_step1_across_plate_stability(df_raw, feature_cols, top_n_to_keep=1500)
df_step1 = df_raw[['Plate', 'Well_ID', 'Treatment'] + stable_cols]

# ==========================================
# STEP 2: T0 NEUTRALITY
# ==========================================
def filter_step2_t0_neutrality(df, features, top_n_to_keep=800):
    t0_df = df[df['Plate'].str.endswith('T0')].copy()
    t0_ctrl_median = t0_df[t0_df['Treatment'] == CONTROL_LABEL][features].median()
    t0_mutants = t0_df[t0_df['Treatment'] != CONTROL_LABEL]
    
    t0_bias_score = (t0_mutants[features] - t0_ctrl_median).abs().mean()
    stable_at_start = t0_bias_score.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_at_start

print("Running Step 2: T0 Neutrality...")
neutral_features = filter_step2_t0_neutrality(df_step1, stable_cols, top_n_to_keep=800)
df_step2 = df_step1[['Plate', 'Well_ID', 'Treatment'] + neutral_features]

# ==========================================
# STEP 3: TEMPORAL STABILITY
# ==========================================
def filter_step3_temporal_stability(df, features, top_n_to_keep=800):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()
    base_plate_names = set([p.split('_')[0] for p in plate_medians.index])
    
    temporal_drifts = []
    for base in base_plate_names:
        p0, p1 = f"{base}_T0", f"{base}_T1"
        if p0 in plate_medians.index and p1 in plate_medians.index:
            drift = (plate_medians.loc[p1] - plate_medians.loc[p0]).abs()
            temporal_drifts.append(drift)
    
    avg_temporal_drift = pd.concat(temporal_drifts, axis=1).mean(axis=1)
    stable_over_time = avg_temporal_drift.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_over_time

print("Running Step 3: Temporal Stability...")
stable_time_features = filter_step3_temporal_stability(df_step2, neutral_features, top_n_to_keep=800)
df_step3 = df_step2[['Plate', 'Well_ID', 'Treatment'] + stable_time_features]

# ==========================================
# STEP 4: AGGRESSIVE PLATE CONVERGENCE
# ==========================================
def filter_step4_plate_convergence(df, features, top_n_to_keep=800):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_noises.append(noise)
    
    avg_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)
    plate_blind_features = avg_plate_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return plate_blind_features

print("Running Step 4: Aggressive Plate Convergence...")
vetted_features_s4 = filter_step4_plate_convergence(df_step3, stable_time_features, top_n_to_keep=800)

# ==========================================
# STEP 4.5: WITHIN-PLATE CONSISTENCY
# ==========================================
def filter_within_plate_consistency(df, features, top_n_to_keep=500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

print("Running Step 4.5: Within-Plate Consistency...")
consistent_features = filter_within_plate_consistency(df_step3, vetted_features_s4, top_n_to_keep=500)
# Update dataframe to reflect survival of 4.5
df_step4_5 = df_step3[['Plate', 'Well_ID', 'Treatment'] + consistent_features]

# ==========================================
# STEP 11: REPEATED ACROSS-PLATE STABILITY
# ==========================================
def filter_step11_plate_convergence(df, features, top_n_to_keep=300):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_noises.append(noise)
    
    avg_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)
    plate_blind_features = avg_plate_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return plate_blind_features

print("Running Step 11: Repeated Aggressive Plate Convergence...")
# Apply Step 11 to the results of Step 4.5
step11_features = filter_step11_plate_convergence(df_step4_5, consistent_features, top_n_to_keep=300)
df_step11 = df_step4_5[['Plate', 'Well_ID', 'Treatment'] + step11_features]

# ==========================================
# STEP 5: REDUNDANCY REMOVAL
# ==========================================
def filter_step5_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

print("Running Step 5: Redundancy Filter...")
non_redundant_features = filter_step5_redundancy(df_step11, step11_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final_vetted = df_step11[['Plate', 'Well_ID', 'Treatment'] + non_redundant_features]
output_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles10.csv")
df_final_vetted.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Final feature count: {len(non_redundant_features)}")
print(f"Master profiles saved to: {output_path}")
print("="*40)

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles10.csv")
df = pd.read_csv(file_path)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x in ["no_sgRNA", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'UMAP1': False, 'UMAP2': False},
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) # Tiny mutants
    fig.update_traces(marker=dict(size=8, line=dict(width=1)), selector=dict(marker_symbol='square')) # Clear controls
    
    fig.update_layout(
        width=1000, 
        height=700,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Show the plot (In Jupyter, these will appear one after another)
    fig.show()
    
    # OPTIONAL: Save to HTML so you can send them to others or open later
    fig.write_html(os.path.join(PROJECT_ROOT, f"whithincorrumap10_{config['name'].replace(' ', '_')}.html"))

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)
feature_cols = [str(i) for i in range(6400)]
df_raw.columns = [str(c) for c in df_raw.columns]

# ==========================================
# STEP 1: ACROSS-PLATE STABILITY
# ==========================================
def filter_step1_across_plate_stability(df, features, top_n_to_keep=1500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    variance_filter = df[features].std() > 0.01
    stable_features = total_batch_noise[variance_filter].sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

print("\nRunning Step 1: Across-Plate Stability...")
stable_cols = filter_step1_across_plate_stability(df_raw, feature_cols, top_n_to_keep=1500)
df_step1 = df_raw[['Plate', 'Well_ID', 'Treatment'] + stable_cols]

# ==========================================
# STEP 2: T0 NEUTRALITY
# ==========================================
def filter_step2_t0_neutrality(df, features, top_n_to_keep=800):
    t0_df = df[df['Plate'].str.endswith('T0')].copy()
    t0_ctrl_median = t0_df[t0_df['Treatment'] == CONTROL_LABEL][features].median()
    t0_mutants = t0_df[t0_df['Treatment'] != CONTROL_LABEL]
    
    t0_bias_score = (t0_mutants[features] - t0_ctrl_median).abs().mean()
    stable_at_start = t0_bias_score.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_at_start

print("Running Step 2: T0 Neutrality...")
neutral_features = filter_step2_t0_neutrality(df_step1, stable_cols, top_n_to_keep=800)
df_step2 = df_step1[['Plate', 'Well_ID', 'Treatment'] + neutral_features]

# ==========================================
# STEP 3: TEMPORAL STABILITY
# ==========================================
def filter_step3_temporal_stability(df, features, top_n_to_keep=800):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()
    base_plate_names = set([p.split('_')[0] for p in plate_medians.index])
    
    temporal_drifts = []
    for base in base_plate_names:
        p0, p1 = f"{base}_T0", f"{base}_T1"
        if p0 in plate_medians.index and p1 in plate_medians.index:
            drift = (plate_medians.loc[p1] - plate_medians.loc[p0]).abs()
            temporal_drifts.append(drift)
    
    avg_temporal_drift = pd.concat(temporal_drifts, axis=1).mean(axis=1)
    stable_over_time = avg_temporal_drift.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_over_time

print("Running Step 3: Temporal Stability...")
stable_time_features = filter_step3_temporal_stability(df_step2, neutral_features, top_n_to_keep=800)
df_step3 = df_step2[['Plate', 'Well_ID', 'Treatment'] + stable_time_features]

# ==========================================
# STEP 4: AGGRESSIVE PLATE CONVERGENCE
# ==========================================
def filter_step4_plate_convergence(df, features, top_n_to_keep=800):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_noises.append(noise)
    
    avg_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)
    plate_blind_features = avg_plate_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return plate_blind_features

print("Running Step 4: Aggressive Plate Convergence...")
vetted_features_s4 = filter_step4_plate_convergence(df_step3, stable_time_features, top_n_to_keep=800)

# ==========================================
# STEP 4.5: WITHIN-PLATE CONSISTENCY
# ==========================================
def filter_within_plate_consistency(df, features, top_n_to_keep=500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

print("Running Step 4.5: Within-Plate Consistency...")
consistent_features = filter_within_plate_consistency(df_step3, vetted_features_s4, top_n_to_keep=500)
# Update dataframe to reflect survival of 4.5
df_step4_5 = df_step3[['Plate', 'Well_ID', 'Treatment'] + consistent_features]

# ==========================================
# STEP 11: REPEATED ACROSS-PLATE STABILITY
# ==========================================
def filter_step11_plate_convergence(df, features, top_n_to_keep=200):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_noises.append(noise)
    
    avg_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)
    plate_blind_features = avg_plate_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return plate_blind_features

print("Running Step 11: Repeated Aggressive Plate Convergence...")
# Apply Step 11 to the results of Step 4.5
step11_features = filter_step11_plate_convergence(df_step4_5, consistent_features, top_n_to_keep=200)
df_step11 = df_step4_5[['Plate', 'Well_ID', 'Treatment'] + step11_features]

# ==========================================
# STEP 5: REDUNDANCY REMOVAL
# ==========================================
def filter_step5_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

print("Running Step 5: Redundancy Filter...")
non_redundant_features = filter_step5_redundancy(df_step11, step11_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final_vetted = df_step11[['Plate', 'Well_ID', 'Treatment'] + non_redundant_features]
output_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles11.csv")
df_final_vetted.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Final feature count: {len(non_redundant_features)}")
print(f"Master profiles saved to: {output_path}")
print("="*40)

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles11.csv")
df = pd.read_csv(file_path)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x in ["no_sgRNA", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'UMAP1': False, 'UMAP2': False},
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) # Tiny mutants
    fig.update_traces(marker=dict(size=8, line=dict(width=1)), selector=dict(marker_symbol='square')) # Clear controls
    
    fig.update_layout(
        width=1000, 
        height=700,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Show the plot (In Jupyter, these will appear one after another)
    fig.show()
    
    # OPTIONAL: Save to HTML so you can send them to others or open later
    fig.write_html(os.path.join(PROJECT_ROOT, f"whithincorrumap10_{config['name'].replace(' ', '_')}.html"))

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)
feature_cols = [str(i) for i in range(6400)]
df_raw.columns = [str(c) for c in df_raw.columns]

# ==========================================
# STEP 1: ACROSS-PLATE STABILITY
# ==========================================
def filter_step1_across_plate_stability(df, features, top_n_to_keep=1500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    variance_filter = df[features].std() > 0.01
    stable_features = total_batch_noise[variance_filter].sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

print("\nRunning Step 1: Across-Plate Stability...")
stable_cols = filter_step1_across_plate_stability(df_raw, feature_cols, top_n_to_keep=1500)
df_step1 = df_raw[['Plate', 'Well_ID', 'Treatment'] + stable_cols]

# ==========================================
# STEP 2: T0 NEUTRALITY
# ==========================================
def filter_step2_t0_neutrality(df, features, top_n_to_keep=1500):
    t0_df = df[df['Plate'].str.endswith('T0')].copy()
    t0_ctrl_median = t0_df[t0_df['Treatment'] == CONTROL_LABEL][features].median()
    t0_mutants = t0_df[t0_df['Treatment'] != CONTROL_LABEL]
    
    t0_bias_score = (t0_mutants[features] - t0_ctrl_median).abs().mean()
    stable_at_start = t0_bias_score.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_at_start

print("Running Step 2: T0 Neutrality...")
neutral_features = filter_step2_t0_neutrality(df_step1, stable_cols, top_n_to_keep=1500)
df_step2 = df_step1[['Plate', 'Well_ID', 'Treatment'] + neutral_features]

# ==========================================
# STEP 3: TEMPORAL STABILITY
# ==========================================
def filter_step3_temporal_stability(df, features, top_n_to_keep=1500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()
    base_plate_names = set([p.split('_')[0] for p in plate_medians.index])
    
    temporal_drifts = []
    for base in base_plate_names:
        p0, p1 = f"{base}_T0", f"{base}_T1"
        if p0 in plate_medians.index and p1 in plate_medians.index:
            drift = (plate_medians.loc[p1] - plate_medians.loc[p0]).abs()
            temporal_drifts.append(drift)
    
    avg_temporal_drift = pd.concat(temporal_drifts, axis=1).mean(axis=1)
    stable_over_time = avg_temporal_drift.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_over_time

print("Running Step 3: Temporal Stability...")
stable_time_features = filter_step3_temporal_stability(df_step2, neutral_features, top_n_to_keep=1500)
df_step3 = df_step2[['Plate', 'Well_ID', 'Treatment'] + stable_time_features]

# ==========================================
# STEP 4: AGGRESSIVE PLATE CONVERGENCE
# ==========================================
def filter_step4_plate_convergence(df, features, top_n_to_keep=1500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_noises.append(noise)
    
    avg_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)
    plate_blind_features = avg_plate_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return plate_blind_features

print("Running Step 4: Aggressive Plate Convergence...")
vetted_features_s4 = filter_step4_plate_convergence(df_step3, stable_time_features, top_n_to_keep=1500)

# ==========================================
# STEP 4.5: WITHIN-PLATE CONSISTENCY
# ==========================================
def filter_within_plate_consistency(df, features, top_n_to_keep=800):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

print("Running Step 4.5: Within-Plate Consistency...")
consistent_features = filter_within_plate_consistency(df_step3, vetted_features_s4, top_n_to_keep=800)
# Update dataframe to reflect survival of 4.5
df_step4_5 = df_step3[['Plate', 'Well_ID', 'Treatment'] + consistent_features]

# ==========================================
# STEP 11: REPEATED ACROSS-PLATE STABILITY
# ==========================================
def filter_step11_plate_convergence(df, features, top_n_to_keep=200):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_noises.append(noise)
    
    avg_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)
    plate_blind_features = avg_plate_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return plate_blind_features

print("Running Step 11: Repeated Aggressive Plate Convergence...")
# Apply Step 11 to the results of Step 4.5
step11_features = filter_step11_plate_convergence(df_step4_5, consistent_features, top_n_to_keep=200)
df_step11 = df_step4_5[['Plate', 'Well_ID', 'Treatment'] + step11_features]

# ==========================================
# STEP 5: REDUNDANCY REMOVAL
# ==========================================
def filter_step5_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

print("Running Step 5: Redundancy Filter...")
non_redundant_features = filter_step5_redundancy(df_step11, step11_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final_vetted = df_step11[['Plate', 'Well_ID', 'Treatment'] + non_redundant_features]
output_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles12.csv")
df_final_vetted.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Final feature count: {len(non_redundant_features)}")
print(f"Master profiles saved to: {output_path}")
print("="*40)

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles12.csv")
df = pd.read_csv(file_path)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x in ["no_sgRNA", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'UMAP1': False, 'UMAP2': False},
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) # Tiny mutants
    fig.update_traces(marker=dict(size=8, line=dict(width=1)), selector=dict(marker_symbol='square')) # Clear controls
    
    fig.update_layout(
        width=1000, 
        height=700,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Show the plot (In Jupyter, these will appear one after another)
    fig.show()
    
    # OPTIONAL: Save to HTML so you can send them to others or open later
    fig.write_html(os.path.join(PROJECT_ROOT, f"whithincorrumap12_{config['name'].replace(' ', '_')}.html"))

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)
feature_cols = [str(i) for i in range(6400)]
df_raw.columns = [str(c) for c in df_raw.columns]

# ==========================================
# STEP 1: ACROSS-PLATE STABILITY
# ==========================================
def filter_step1_across_plate_stability(df, features, top_n_to_keep=2000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    variance_filter = df[features].std() > 0.01
    stable_features = total_batch_noise[variance_filter].sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

print("\nRunning Step 1: Across-Plate Stability...")
stable_cols = filter_step1_across_plate_stability(df_raw, feature_cols, top_n_to_keep=2000)
df_step1 = df_raw[['Plate', 'Well_ID', 'Treatment'] + stable_cols]

# ==========================================
# STEP 2: T0 NEUTRALITY
# ==========================================
def filter_step2_t0_neutrality(df, features, top_n_to_keep=2000):
    t0_df = df[df['Plate'].str.endswith('T0')].copy()
    t0_ctrl_median = t0_df[t0_df['Treatment'] == CONTROL_LABEL][features].median()
    t0_mutants = t0_df[t0_df['Treatment'] != CONTROL_LABEL]
    
    t0_bias_score = (t0_mutants[features] - t0_ctrl_median).abs().mean()
    stable_at_start = t0_bias_score.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_at_start

print("Running Step 2: T0 Neutrality...")
neutral_features = filter_step2_t0_neutrality(df_step1, stable_cols, top_n_to_keep=2000)
df_step2 = df_step1[['Plate', 'Well_ID', 'Treatment'] + neutral_features]

# ==========================================
# STEP 3: TEMPORAL STABILITY
# ==========================================
def filter_step3_temporal_stability(df, features, top_n_to_keep=2000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()
    base_plate_names = set([p.split('_')[0] for p in plate_medians.index])
    
    temporal_drifts = []
    for base in base_plate_names:
        p0, p1 = f"{base}_T0", f"{base}_T1"
        if p0 in plate_medians.index and p1 in plate_medians.index:
            drift = (plate_medians.loc[p1] - plate_medians.loc[p0]).abs()
            temporal_drifts.append(drift)
    
    avg_temporal_drift = pd.concat(temporal_drifts, axis=1).mean(axis=1)
    stable_over_time = avg_temporal_drift.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_over_time

print("Running Step 3: Temporal Stability...")
stable_time_features = filter_step3_temporal_stability(df_step2, neutral_features, top_n_to_keep=2000)
df_step3 = df_step2[['Plate', 'Well_ID', 'Treatment'] + stable_time_features]

# ==========================================
# STEP 4: AGGRESSIVE PLATE CONVERGENCE
# ==========================================
def filter_step4_plate_convergence(df, features, top_n_to_keep=2000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_noises.append(noise)
    
    avg_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)
    plate_blind_features = avg_plate_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return plate_blind_features

print("Running Step 4: Aggressive Plate Convergence...")
vetted_features_s4 = filter_step4_plate_convergence(df_step3, stable_time_features, top_n_to_keep=2000)

# ==========================================
# STEP 4.5: WITHIN-PLATE CONSISTENCY
# ==========================================
def filter_within_plate_consistency(df, features, top_n_to_keep=1000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

print("Running Step 4.5: Within-Plate Consistency...")
consistent_features = filter_within_plate_consistency(df_step3, vetted_features_s4, top_n_to_keep=1000)
# Update dataframe to reflect survival of 4.5
df_step4_5 = df_step3[['Plate', 'Well_ID', 'Treatment'] + consistent_features]

# ==========================================
# STEP 11: REPEATED ACROSS-PLATE STABILITY
# ==========================================
def filter_step11_plate_convergence(df, features, top_n_to_keep=500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_noises.append(noise)
    
    avg_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)
    plate_blind_features = avg_plate_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return plate_blind_features

print("Running Step 11: Repeated Aggressive Plate Convergence...")
# Apply Step 11 to the results of Step 4.5
step11_features = filter_step11_plate_convergence(df_step4_5, consistent_features, top_n_to_keep=500)
df_step11 = df_step4_5[['Plate', 'Well_ID', 'Treatment'] + step11_features]

# ==========================================
# STEP 5: REDUNDANCY REMOVAL
# ==========================================
def filter_step5_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

print("Running Step 5: Redundancy Filter...")
non_redundant_features = filter_step5_redundancy(df_step11, step11_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final_vetted = df_step11[['Plate', 'Well_ID', 'Treatment'] + non_redundant_features]
output_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles13.csv")
df_final_vetted.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Final feature count: {len(non_redundant_features)}")
print(f"Master profiles saved to: {output_path}")
print("="*40)

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles13.csv")
df = pd.read_csv(file_path)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x in ["no_sgRNA", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'UMAP1': False, 'UMAP2': False},
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) # Tiny mutants
    fig.update_traces(marker=dict(size=8, line=dict(width=1)), selector=dict(marker_symbol='square')) # Clear controls
    
    fig.update_layout(
        width=1000, 
        height=700,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Show the plot (In Jupyter, these will appear one after another)
    fig.show()
    
    # OPTIONAL: Save to HTML so you can send them to others or open later
    fig.write_html(os.path.join(PROJECT_ROOT, f"whithincorrumap13_{config['name'].replace(' ', '_')}.html"))

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)
feature_cols = [str(i) for i in range(6400)]
df_raw.columns = [str(c) for c in df_raw.columns]

# ==========================================
# STEP 1: ACROSS-PLATE STABILITY
# ==========================================
def filter_step1_across_plate_stability(df, features, top_n_to_keep=2000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    variance_filter = df[features].std() > 0.01
    stable_features = total_batch_noise[variance_filter].sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

print("\nRunning Step 1: Across-Plate Stability...")
stable_cols = filter_step1_across_plate_stability(df_raw, feature_cols, top_n_to_keep=2000)
df_step1 = df_raw[['Plate', 'Well_ID', 'Treatment'] + stable_cols]

# ==========================================
# STEP 2: T0 NEUTRALITY
# ==========================================
def filter_step2_t0_neutrality(df, features, top_n_to_keep=2000):
    t0_df = df[df['Plate'].str.endswith('T0')].copy()
    t0_ctrl_median = t0_df[t0_df['Treatment'] == CONTROL_LABEL][features].median()
    t0_mutants = t0_df[t0_df['Treatment'] != CONTROL_LABEL]
    
    t0_bias_score = (t0_mutants[features] - t0_ctrl_median).abs().mean()
    stable_at_start = t0_bias_score.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_at_start

print("Running Step 2: T0 Neutrality...")
neutral_features = filter_step2_t0_neutrality(df_step1, stable_cols, top_n_to_keep=2000)
df_step2 = df_step1[['Plate', 'Well_ID', 'Treatment'] + neutral_features]

# ==========================================
# STEP 3: TEMPORAL STABILITY
# ==========================================
def filter_step3_temporal_stability(df, features, top_n_to_keep=2000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()
    base_plate_names = set([p.split('_')[0] for p in plate_medians.index])
    
    temporal_drifts = []
    for base in base_plate_names:
        p0, p1 = f"{base}_T0", f"{base}_T1"
        if p0 in plate_medians.index and p1 in plate_medians.index:
            drift = (plate_medians.loc[p1] - plate_medians.loc[p0]).abs()
            temporal_drifts.append(drift)
    
    avg_temporal_drift = pd.concat(temporal_drifts, axis=1).mean(axis=1)
    stable_over_time = avg_temporal_drift.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_over_time

print("Running Step 3: Temporal Stability...")
stable_time_features = filter_step3_temporal_stability(df_step2, neutral_features, top_n_to_keep=2000)
df_step3 = df_step2[['Plate', 'Well_ID', 'Treatment'] + stable_time_features]

# ==========================================
# STEP 4: AGGRESSIVE PLATE CONVERGENCE
# ==========================================
def filter_step4_plate_convergence(df, features, top_n_to_keep=2000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_noises.append(noise)
    
    avg_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)
    plate_blind_features = avg_plate_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return plate_blind_features

print("Running Step 4: Aggressive Plate Convergence...")
vetted_features_s4 = filter_step4_plate_convergence(df_step3, stable_time_features, top_n_to_keep=2000)

# ==========================================
# STEP 4.5: WITHIN-PLATE CONSISTENCY
# ==========================================
def filter_within_plate_consistency(df, features, top_n_to_keep=1000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

print("Running Step 4.5: Within-Plate Consistency...")
consistent_features = filter_within_plate_consistency(df_step3, vetted_features_s4, top_n_to_keep=1000)
# Update dataframe to reflect survival of 4.5
df_step4_5 = df_step3[['Plate', 'Well_ID', 'Treatment'] + consistent_features]

# ==========================================
# STEP 11: REPEATED ACROSS-PLATE STABILITY
# ==========================================
def filter_step11_plate_convergence(df, features, top_n_to_keep=50):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_noises.append(noise)
    
    avg_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)
    plate_blind_features = avg_plate_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return plate_blind_features

print("Running Step 11: Repeated Aggressive Plate Convergence...")
# Apply Step 11 to the results of Step 4.5
step11_features = filter_step11_plate_convergence(df_step4_5, consistent_features, top_n_to_keep=50)
df_step11 = df_step4_5[['Plate', 'Well_ID', 'Treatment'] + step11_features]

# ==========================================
# STEP 5: REDUNDANCY REMOVAL
# ==========================================
def filter_step5_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

print("Running Step 5: Redundancy Filter...")
non_redundant_features = filter_step5_redundancy(df_step11, step11_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final_vetted = df_step11[['Plate', 'Well_ID', 'Treatment'] + non_redundant_features]
output_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles15.csv")
df_final_vetted.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Final feature count: {len(non_redundant_features)}")
print(f"Master profiles saved to: {output_path}")
print("="*40)

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles15.csv")
df = pd.read_csv(file_path)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x in ["no_sgRNA", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'UMAP1': False, 'UMAP2': False},
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) # Tiny mutants
    fig.update_traces(marker=dict(size=8, line=dict(width=1)), selector=dict(marker_symbol='square')) # Clear controls
    
    fig.update_layout(
        width=1000, 
        height=700,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Show the plot (In Jupyter, these will appear one after another)
    fig.show()
    
    # OPTIONAL: Save to HTML so you can send them to others or open later
    fig.write_html(os.path.join(PROJECT_ROOT, f"whithincorrumap15_{config['name'].replace(' ', '_')}.html"))

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)
feature_cols = [str(i) for i in range(6400)]
df_raw.columns = [str(c) for c in df_raw.columns]

# ==========================================
# STEP 1: ACROSS-PLATE STABILITY
# ==========================================
def filter_step1_across_plate_stability(df, features, top_n_to_keep=2000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    variance_filter = df[features].std() > 0.01
    stable_features = total_batch_noise[variance_filter].sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

print("\nRunning Step 1: Across-Plate Stability...")
stable_cols = filter_step1_across_plate_stability(df_raw, feature_cols, top_n_to_keep=2000)
df_step1 = df_raw[['Plate', 'Well_ID', 'Treatment'] + stable_cols]

# ==========================================
# STEP 2: T0 NEUTRALITY
# ==========================================
def filter_step2_t0_neutrality(df, features, top_n_to_keep=2000):
    t0_df = df[df['Plate'].str.endswith('T0')].copy()
    t0_ctrl_median = t0_df[t0_df['Treatment'] == CONTROL_LABEL][features].median()
    t0_mutants = t0_df[t0_df['Treatment'] != CONTROL_LABEL]
    
    t0_bias_score = (t0_mutants[features] - t0_ctrl_median).abs().mean()
    stable_at_start = t0_bias_score.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_at_start

print("Running Step 2: T0 Neutrality...")
neutral_features = filter_step2_t0_neutrality(df_step1, stable_cols, top_n_to_keep=2000)
df_step2 = df_step1[['Plate', 'Well_ID', 'Treatment'] + neutral_features]

# ==========================================
# STEP 3: TEMPORAL STABILITY
# ==========================================
def filter_step3_temporal_stability(df, features, top_n_to_keep=2000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()
    base_plate_names = set([p.split('_')[0] for p in plate_medians.index])
    
    temporal_drifts = []
    for base in base_plate_names:
        p0, p1 = f"{base}_T0", f"{base}_T1"
        if p0 in plate_medians.index and p1 in plate_medians.index:
            drift = (plate_medians.loc[p1] - plate_medians.loc[p0]).abs()
            temporal_drifts.append(drift)
    
    avg_temporal_drift = pd.concat(temporal_drifts, axis=1).mean(axis=1)
    stable_over_time = avg_temporal_drift.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_over_time

print("Running Step 3: Temporal Stability...")
stable_time_features = filter_step3_temporal_stability(df_step2, neutral_features, top_n_to_keep=2000)
df_step3 = df_step2[['Plate', 'Well_ID', 'Treatment'] + stable_time_features]

# ==========================================
# STEP 4: AGGRESSIVE PLATE CONVERGENCE
# ==========================================
def filter_step4_plate_convergence(df, features, top_n_to_keep=2000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_noises.append(noise)
    
    avg_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)
    plate_blind_features = avg_plate_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return plate_blind_features

print("Running Step 4: Aggressive Plate Convergence...")
vetted_features_s4 = filter_step4_plate_convergence(df_step3, stable_time_features, top_n_to_keep=2000)

# ==========================================
# STEP 4.5: WITHIN-PLATE CONSISTENCY
# ==========================================
def filter_within_plate_consistency(df, features, top_n_to_keep=1000):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

print("Running Step 4.5: Within-Plate Consistency...")
consistent_features = filter_within_plate_consistency(df_step3, vetted_features_s4, top_n_to_keep=1000)
# Update dataframe to reflect survival of 4.5
df_step4_5 = df_step3[['Plate', 'Well_ID', 'Treatment'] + consistent_features]

# ==========================================
# STEP 11: REPEATED ACROSS-PLATE STABILITY
# ==========================================
def filter_step11_plate_convergence(df, features, top_n_to_keep=20):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_noises.append(noise)
    
    avg_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)
    plate_blind_features = avg_plate_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return plate_blind_features

print("Running Step 11: Repeated Aggressive Plate Convergence...")
# Apply Step 11 to the results of Step 4.5
step11_features = filter_step11_plate_convergence(df_step4_5, consistent_features, top_n_to_keep=20)
df_step11 = df_step4_5[['Plate', 'Well_ID', 'Treatment'] + step11_features]

# ==========================================
# STEP 5: REDUNDANCY REMOVAL
# ==========================================
def filter_step5_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

print("Running Step 5: Redundancy Filter...")
non_redundant_features = filter_step5_redundancy(df_step11, step11_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final_vetted = df_step11[['Plate', 'Well_ID', 'Treatment'] + non_redundant_features]
output_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles16.csv")
df_final_vetted.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Final feature count: {len(non_redundant_features)}")
print(f"Master profiles saved to: {output_path}")
print("="*40)

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles16.csv")
df = pd.read_csv(file_path)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x in ["no_sgRNA", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'UMAP1': False, 'UMAP2': False},
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) # Tiny mutants
    fig.update_traces(marker=dict(size=8, line=dict(width=1)), selector=dict(marker_symbol='square')) # Clear controls
    
    fig.update_layout(
        width=1000, 
        height=700,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Show the plot (In Jupyter, these will appear one after another)
    fig.show()
    
    # OPTIONAL: Save to HTML so you can send them to others or open later
    fig.write_html(os.path.join(PROJECT_ROOT, f"whithincorrumap16_{config['name'].replace(' ', '_')}.html"))

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. SETUP
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")
CONTROL_LABEL = "no_sgRNA" 

df = pd.read_csv(INPUT_CSV)
feature_cols = [str(i) for i in range(6400)]
df.columns = [str(c) for c in df.columns]
df['TP'] = df['Plate'].str.extract(r'_(T\d)')

# ==========================================
# STEP A: THE NOISE (T0 Plate Variance)
# ==========================================
# We calculate how much plates disagree at T0 (Baseline)
t0_ctrls = df[(df['TP'] == 'T0') & (df['Treatment'] == CONTROL_LABEL)]
plate_noise_t0 = t0_ctrls.groupby('Plate')[feature_cols].median().std()

# ==========================================
# STEP B: THE SIGNAL (Max Spread at T1 or T2)
# ==========================================
# We calculate the biological spread at T1 and T2 separately
t1_spread = df[df['TP'] == 'T1'][feature_cols].std()
t2_spread = df[df['TP'] == 'T2'][feature_cols].std()

# We take the MAXIMUM spread across T1 and T2 
# This protects both early (T1) and late (T2) phenotypes
max_bio_signal = pd.concat([t1_spread, t2_spread], axis=1).max(axis=1)

# ==========================================
# STEP C: SELECTION (Signal-to-Noise Ratio)
# ==========================================
# High Signal (at either T1 or T2) / Low Noise (at T0)
snr_score = max_bio_signal / (plate_noise_t0 + 1e-6)

# Basic variance filter to remove "dead" features
variance_filter = df[feature_cols].std() > 0.01
valid_snr = snr_score[variance_filter]

# Keep the top 300 features
vetted_features = valid_snr.sort_values(ascending=False).head(300).index.tolist()

# ==========================================
# STEP D: REDUNDANCY & SAVE
# ==========================================
df_vetted = df[['Plate', 'Well_ID', 'Treatment'] + vetted_features]
corr_matrix = df_vetted[vetted_features].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
non_redundant = [f for f in vetted_features if f not in to_drop]

output_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles_unbiased111.csv")
df[['Plate', 'Well_ID', 'Treatment'] + non_redundant].to_csv(output_path, index=False)

print(f"Selection Complete: {len(non_redundant)} features.")
print("Logic: Balanced for early (T1) and late (T2) biological signals while forcing T0 convergence.")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles_unbiased111.csv")
df = pd.read_csv(file_path)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x in ["no_sgRNA", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'UMAP1': False, 'UMAP2': False},
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) # Tiny mutants
    fig.update_traces(marker=dict(size=8, line=dict(width=1)), selector=dict(marker_symbol='square')) # Clear controls
    
    fig.update_layout(
        width=1000, 
        height=700,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Show the plot (In Jupyter, these will appear one after another)
    fig.show()
    
    # OPTIONAL: Save to HTML so you can send them to others or open later
    fig.write_html(os.path.join(PROJECT_ROOT, f"whithincorrumap17_{config['name'].replace(' ', '_')}.html"))

In [ ]:
#final easiest

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)
feature_cols = [str(i) for i in range(6400)]
df_raw.columns = [str(c) for c in df_raw.columns]

# ==========================================
# STEP 1: ACROSS-PLATE STABILITY
# ==========================================
def filter_step1_across_plate_stability(df, features, top_n_to_keep=800):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    variance_filter = df[features].std() > 0.01
    stable_features = total_batch_noise[variance_filter].sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

print("\nRunning Step 1: Across-Plate Stability...")
stable_cols = filter_step1_across_plate_stability(df_raw, feature_cols, top_n_to_keep=800)
df_step1 = df_raw[['Plate', 'Well_ID', 'Treatment'] + stable_cols]

# ==========================================
# STEP 2: T0 NEUTRALITY
# ==========================================
def filter_step2_t0_neutrality(df, features, top_n_to_keep=800):
    t0_df = df[df['Plate'].str.endswith('T0')].copy()
    t0_ctrl_median = t0_df[t0_df['Treatment'] == CONTROL_LABEL][features].median()
    t0_mutants = t0_df[t0_df['Treatment'] != CONTROL_LABEL]
    
    t0_bias_score = (t0_mutants[features] - t0_ctrl_median).abs().mean()
    stable_at_start = t0_bias_score.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_at_start

print("Running Step 2: T0 Neutrality...")
neutral_features = filter_step2_t0_neutrality(df_step1, stable_cols, top_n_to_keep=800)
df_step2 = df_step1[['Plate', 'Well_ID', 'Treatment'] + neutral_features]

# ==========================================
# STEP 3: TEMPORAL STABILITY
# ==========================================
def filter_step3_temporal_stability(df, features, top_n_to_keep=800):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()
    base_plate_names = set([p.split('_')[0] for p in plate_medians.index])
    
    temporal_drifts = []
    for base in base_plate_names:
        p0, p1 = f"{base}_T0", f"{base}_T1"
        if p0 in plate_medians.index and p1 in plate_medians.index:
            drift = (plate_medians.loc[p1] - plate_medians.loc[p0]).abs()
            temporal_drifts.append(drift)
    
    avg_temporal_drift = pd.concat(temporal_drifts, axis=1).mean(axis=1)
    stable_over_time = avg_temporal_drift.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_over_time

print("Running Step 3: Temporal Stability...")
stable_time_features = filter_step3_temporal_stability(df_step2, neutral_features, top_n_to_keep=800)
df_step3 = df_step2[['Plate', 'Well_ID', 'Treatment'] + stable_time_features]

# ==========================================
# STEP 4: AGGRESSIVE PLATE CONVERGENCE
# ==========================================
def filter_step4_plate_convergence(df, features, top_n_to_keep=800):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_noises.append(noise)
    
    avg_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)
    plate_blind_features = avg_plate_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return plate_blind_features

print("Running Step 4: Aggressive Plate Convergence...")
vetted_features_s4 = filter_step4_plate_convergence(df_step3, stable_time_features, top_n_to_keep=800)

# ==========================================
# STEP 4.5: WITHIN-PLATE CONSISTENCY
# ==========================================
def filter_within_plate_consistency(df, features, top_n_to_keep=500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

print("Running Step 4.5: Within-Plate Consistency...")
consistent_features = filter_within_plate_consistency(df_step3, vetted_features_s4, top_n_to_keep=500)
# Update dataframe to reflect survival of 4.5
df_step4_5 = df_step3[['Plate', 'Well_ID', 'Treatment'] + consistent_features]

# ==========================================
# STEP 11: REPEATED ACROSS-PLATE STABILITY
# ==========================================
def filter_step11_plate_convergence(df, features, top_n_to_keep=200):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_noises.append(noise)
    
    avg_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)
    plate_blind_features = avg_plate_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return plate_blind_features

print("Running Step 11: Repeated Aggressive Plate Convergence...")
# Apply Step 11 to the results of Step 4.5
step11_features = filter_step11_plate_convergence(df_step4_5, consistent_features, top_n_to_keep=200)
df_step11 = df_step4_5[['Plate', 'Well_ID', 'Treatment'] + step11_features]

# ==========================================
# STEP 5: REDUNDANCY REMOVAL
# ==========================================
def filter_step5_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

print("Running Step 5: Redundancy Filter...")
non_redundant_features = filter_step5_redundancy(df_step11, step11_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final_vetted = df_step11[['Plate', 'Well_ID', 'Treatment'] + non_redundant_features]
output_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles20.csv")
df_final_vetted.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Final feature count: {len(non_redundant_features)}")
print(f"Master profiles saved to: {output_path}")
print("="*40)

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles20.csv")
df = pd.read_csv(file_path)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x in ["no_sgRNA", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'UMAP1': False, 'UMAP2': False},
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) # Tiny mutants
    fig.update_traces(marker=dict(size=8, line=dict(width=1)), selector=dict(marker_symbol='square')) # Clear controls
    
    fig.update_layout(
        width=1000, 
        height=700,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Show the plot (In Jupyter, these will appear one after another)
    fig.show()
    
    # OPTIONAL: Save to HTML so you can send them to others or open later
    fig.write_html(os.path.join(PROJECT_ROOT, f"whithincorrumap20_{config['name'].replace(' ', '_')}.html"))

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles20.csv")
df = pd.read_csv(file_path)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T2","PLATE2_T2","PLATE3_T2","PLATE4_T2",
                   "PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x in ["no_sgRNA", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'UMAP1': False, 'UMAP2': False},
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) # Tiny mutants
    fig.update_traces(marker=dict(size=8, line=dict(width=1)), selector=dict(marker_symbol='square')) # Clear controls
    
    fig.update_layout(
        width=1000, 
        height=700,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Show the plot (In Jupyter, these will appear one after another)
    fig.show()
    
    # OPTIONAL: Save to HTML so you can send them to others or open later
    fig.write_html(os.path.join(PROJECT_ROOT, f"whithincorrumap20_2_{config['name'].replace(' ', '_')}.html"))

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles20.csv")
df = pd.read_csv(file_path)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE2_T0","PLATE3_T0","PLATE4_T0",
                   "PLATE5_T0"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x in ["no_sgRNA", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'UMAP1': False, 'UMAP2': False},
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) # Tiny mutants
    fig.update_traces(marker=dict(size=8, line=dict(width=1)), selector=dict(marker_symbol='square')) # Clear controls
    
    fig.update_layout(
        width=1000, 
        height=700,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Show the plot (In Jupyter, these will appear one after another)
    fig.show()
    
    # OPTIONAL: Save to HTML so you can send them to others or open later
    fig.write_html(os.path.join(PROJECT_ROOT, f"whithincorrumap20_0_{config['name'].replace(' ', '_')}.html"))

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# --- 1. SETUP ---
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "UMAP_20_annotated")

# Create the output directory if it doesn't exist
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles20.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

df = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df.columns = [str(c) for c in df.columns]

# --- 2. MERGE & FIX ANNOTATIONS ---
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
df['SubtiWiki Annotation 4'] = df['SubtiWiki Annotation 4'].fillna("Unknown/Other")
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

df['Display_Category'] = df['SubtiWiki Annotation 4']
df.loc[df['Timepoint'] == 'T0', 'Display_Category'] = 'Baseline (T0)'

# --- 3. DYNAMIC COLOR MAPPING ---
all_cats = df['Display_Category'].unique()
standard_colors = px.colors.qualitative.Alphabet 

color_map = {cat: standard_colors[i % len(standard_colors)] for i, cat in enumerate(all_cats)}
color_map['Baseline (T0)'] = 'lightgrey'
color_map['Unknown/Other'] = 'black'

# --- 4. RUN UMAP LOOP AND SAVE ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Vetted_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Generating and Saving UMAP for: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    fig = px.scatter(
        df, x='UMAP1', y='UMAP2', 
        color='Display_Category',
        symbol='Timepoint',
        symbol_map={"T0": "circle", "T1": "x", "T2": "circle"},
        hover_name='Treatment',
        hover_data=['Plate', 'SubtiWiki Annotation 4'],
        title=f"Pathway Overlay: {config['name'].replace('_', ' ')}",
        color_discrete_map=color_map,
        template='plotly_white'
    )
    
    fig.update_traces(marker=dict(opacity=1.0)) 
    fig.update_traces(marker=dict(size=5), selector=dict(marker_symbol='circle'))
    fig.update_traces(marker=dict(size=6), selector=dict(marker_symbol='x'))
    
    fig.update_layout(width=1100, height=700)
    
    # --- SAVE TO LOCATION ---
    save_path = os.path.join(OUTPUT_DIR, f"UMAP_{config['name']}.html")
    fig.write_html(save_path)
    
    # Optional: also show in the notebook
    fig.show()

print(f"\nAll interactive plots have been saved to: {OUTPUT_DIR}")

In [ ]:
#metmutantnamen erop
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "UMAP_20_annotated")

# Create the output directory if it doesn't exist
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles20.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

df = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df.columns = [str(c) for c in df.columns]

# --- 2. MERGE & FIX ANNOTATIONS ---
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
df['SubtiWiki Annotation 4'] = df['SubtiWiki Annotation 4'].fillna("Unknown/Other")
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

df['Display_Category'] = df['SubtiWiki Annotation 4']
df.loc[df['Timepoint'] == 'T0', 'Display_Category'] = 'Baseline (T0)'

# --- 3. DYNAMIC COLOR MAPPING ---
all_cats = df['Display_Category'].unique()
standard_colors = px.colors.qualitative.Alphabet 

color_map = {cat: standard_colors[i % len(standard_colors)] for i, cat in enumerate(all_cats)}
color_map['Baseline (T0)'] = 'lightgrey'
color_map['Unknown/Other'] = 'black'


# --- 4. RUN UMAP LOOP AND SAVE ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Vetted_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Generating and Saving UMAP for: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    fig = px.scatter(
        df, x='UMAP1', y='UMAP2', 
        color='Display_Category',
        symbol='Timepoint',
        # ADDED TEXT PARAMETER HERE
        text='Treatment', 
        symbol_map={"T0": "circle", "T1": "x", "T2": "circle"},
        hover_name='Treatment',
        hover_data=['Plate', 'SubtiWiki Annotation 4'],
        title=f"Pathway Overlay: {config['name'].replace('_', ' ')}",
        color_discrete_map=color_map,
        template='plotly_white'
    )
    
    # UPDATED TRACES: Set mode to show both markers and text
    # textposition ensures the name doesn't sit directly on top of the dot
    fig.update_traces(
        mode='markers+text', 
        textposition='top center',
        marker=dict(opacity=1.0)
    ) 
    
    fig.update_traces(marker=dict(size=5), selector=dict(marker_symbol='circle'))
    fig.update_traces(marker=dict(size=6), selector=dict(marker_symbol='x'))
    
    # Optional: Hide text by default for 'Baseline (T0)' to reduce clutter
    # You can comment these lines out if you want to see T0 names too
    fig.for_each_trace(
        lambda t: t.update(textfont_color='rgba(0,0,0,0)') if t.name == 'Baseline (T0)' else None
    )

    fig.update_layout(width=1100, height=700)
    
    # --- SAVE TO LOCATION ---
    save_path = os.path.join(OUTPUT_DIR, f"UMAPmetmutantnaam_{config['name']}.html")
    fig.write_html(save_path)
    
    fig.show()

print(f"\nAll interactive plots have been saved to: {OUTPUT_DIR}")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "UMAP_MultiShade_Plots_20")
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles20.csv")
df = pd.read_csv(file_path)
df.columns = [str(c) for c in df.columns]

# 2. DEFINE THE NESTED COLOR MAP
# We manually define shades for each Plate/Timepoint combination
nested_color_map = {
    # T0 - The Blue Family (Baseline)
    "PLATE1_T0": "#DEEBF7", "PLATE2_T0": "#C6DBEF", "PLATE3_T0": "#9ECAE1", 
    "PLATE4_T0": "#6BAED6", "PLATE5_T0": "#4292C6",
    
    # T1 - The Orange/Red Family (Early Response)
    "PLATE1_T1": "#FEE6CE", "PLATE2_T1": "#FDD0A2", "PLATE3_T1": "#FDAE6B", 
    "PLATE4_T1": "#FD8D3C", "PLATE5_T1": "#F16913",
    
    # T2 - The Purple/Dark Family (Final Phenotype)
    "PLATE1_T2": "#EFEDF5", "PLATE2_T2": "#DADAEB", "PLATE3_T2": "#BCBDDC", 
    "PLATE4_T2": "#9E9AC8", "PLATE5_T2": "#807DBA"
}

# Create a 'Type' column for shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x in ["no_sgRNA", "nosgrna"] else 'Mutant')

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# 4. RUN UMAP LOOP
for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Processing Multi-Shade UMAP for: {config['name']}...")
    
    # Reduce
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df_plot = df.copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    # Sort for consistent layering (T0 at back, T2 at front)
    df_plot['Timepoint_Rank'] = df_plot['Plate'].str.extract(r'_(T\d)')[0]
    df_plot = df_plot.sort_values('Timepoint_Rank')

    fig = px.scatter(
        df_plot, 
        x='UMAP1', y='UMAP2', 
        color='Plate',
        color_discrete_map=nested_color_map, 
        symbol='Type',
        symbol_map={'Control': 'square', 'Mutant': 'circle'},
        hover_name='Treatment',
        hover_data=['Plate', 'Well_ID'],
        title=f"Multi-Shade Timepoint UMAP: {config['name']}",
        template='plotly_white'
    )
    
    # Visual Density Tweaks
    fig.update_traces(marker=dict(size=5, opacity=0.8), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=8, line=dict(width=1, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=1100, 
        height=800, 
        legend_title_text='Plate & Timepoint Gradient'
    )
    
    # SAVE & SHOW
    save_name = config['name'].replace(' ', '_')
    fig.write_html(os.path.join(OUTPUT_DIR, f"multishade_umap_{save_name}.html"))
    fig.show()

print(f"Success! Multi-shade plots saved to: {OUTPUT_DIR}")

In [ ]:
#check of hetzelfde

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# --- 1. SETUP ---
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "UMAP_20_annotated")

# Create the output directory if it doesn't exist
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles20.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

df = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df.columns = [str(c) for c in df.columns]

# --- 2. MERGE & FIX ANNOTATIONS ---
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
df['SubtiWiki Annotation 4'] = df['SubtiWiki Annotation 4'].fillna("Unknown/Other")
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

df['Display_Category'] = df['SubtiWiki Annotation 4']
df.loc[df['Timepoint'] == 'T0', 'Display_Category'] = 'Baseline (T0)'

# --- 3. DYNAMIC COLOR MAPPING ---
all_cats = df['Display_Category'].unique()
standard_colors = px.colors.qualitative.Alphabet 

color_map = {cat: standard_colors[i % len(standard_colors)] for i, cat in enumerate(all_cats)}
color_map['Baseline (T0)'] = 'lightgrey'
color_map['Unknown/Other'] = 'black'



In [ ]:
# --- 4. RUN UMAP LOOP AND SAVE ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Vetted_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Generating and Saving UMAP for: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    fig = px.scatter(
        df, x='UMAP1', y='UMAP2', 
        color='Display_Category',
        symbol='Timepoint',
        symbol_map={"T0": "circle", "T1": "x", "T2": "circle"},
        hover_name='Treatment',
        hover_data=['Plate', 'SubtiWiki Annotation 4'],
        title=f"Pathway Overlay: {config['name'].replace('_', ' ')}",
        color_discrete_map=color_map,
        template='plotly_white'
    )
    
    fig.update_traces(marker=dict(opacity=1.0)) 
    fig.update_traces(marker=dict(size=5), selector=dict(marker_symbol='circle'))
    fig.update_traces(marker=dict(size=6), selector=dict(marker_symbol='x'))
    
    fig.update_layout(width=1100, height=700)
    
    # --- SAVE TO LOCATION ---
    save_path = os.path.join(OUTPUT_DIR, f"UMAPenkelannotated_{config['name']}.html")
    fig.write_html(save_path)
    
    # Optional: also show in the notebook
    fig.show()

print(f"\nAll interactive plots have been saved to: {OUTPUT_DIR}")

In [ ]:
#annotated met naaam

# --- 4. RUN UMAP LOOP AND SAVE ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Vetted_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Generating and Saving UMAP for: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    fig = px.scatter(
        df, x='UMAP1', y='UMAP2', 
        color='Display_Category',
        symbol='Timepoint',
        # ADDED TEXT PARAMETER HERE
        text='Treatment', 
        symbol_map={"T0": "circle", "T1": "x", "T2": "circle"},
        hover_name='Treatment',
        hover_data=['Plate', 'SubtiWiki Annotation 4'],
        title=f"Pathway Overlay: {config['name'].replace('_', ' ')}",
        color_discrete_map=color_map,
        template='plotly_white'
    )
    
    # UPDATED TRACES: Set mode to show both markers and text
    # textposition ensures the name doesn't sit directly on top of the dot
    fig.update_traces(
        mode='markers+text', 
        textposition='top center',
        marker=dict(opacity=1.0)
    ) 
    
    fig.update_traces(marker=dict(size=5), selector=dict(marker_symbol='circle'))
    fig.update_traces(marker=dict(size=6), selector=dict(marker_symbol='x'))
    
    # Optional: Hide text by default for 'Baseline (T0)' to reduce clutter
    # You can comment these lines out if you want to see T0 names too
    fig.for_each_trace(
        lambda t: t.update(textfont_color='rgba(0,0,0,0)') if t.name == 'Baseline (T0)' else None
    )

    fig.update_layout(width=1100, height=700)
    
    # --- SAVE TO LOCATION ---
    save_path = os.path.join(OUTPUT_DIR, f"UMAPannotatedmetmutantnaam_{config['name']}.html")
    fig.write_html(save_path)
    
    fig.show()

print(f"\nAll interactive plots have been saved to: {OUTPUT_DIR}")

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)

# Ensure all columns are strings to avoid indexing issues
df_raw.columns = [str(c) for c in df_raw.columns]
feature_cols = [str(i) for i in range(6400)]

# ==========================================
# STEP 1: ACROSS-PLATE STABILITY
# ==========================================
def filter_step1_across_plate_stability(df, features, top_n_to_keep=800):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    # Basic variance check to remove dead/constant features
    variance_filter = df[features].std() > 0.01
    
    stable_features = total_batch_noise[variance_filter].sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

print("\nRunning Step 1: Across-Plate Stability...")
stable_cols = filter_step1_across_plate_stability(df_raw, feature_cols, top_n_to_keep=800)
df_step1 = df_raw[['Plate', 'Well_ID', 'Treatment'] + stable_cols]

# ==========================================
# STEP 2: WITHIN-PLATE CONSISTENCY (Former 4.5)
# ==========================================
def filter_within_plate_consistency(df, features, top_n_to_keep=500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    # Calculate standard deviation within each plate for controls
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

print("Running Step 2: Within-Plate Consistency...")
# FIX: Changed 'vetted_features_s4' to 'stable_cols'
consistent_features = filter_within_plate_consistency(df_step1, stable_cols, top_n_to_keep=500)
df_step2 = df_step1[['Plate', 'Well_ID', 'Treatment'] + consistent_features]

# ==========================================
# STEP 3: FINAL PLATE CONVERGENCE (Former 11)
# ==========================================
def filter_step3_plate_convergence(df, features, top_n_to_keep=200):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_noises.append(noise)
    
    avg_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)
    plate_blind_features = avg_plate_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return plate_blind_features

print("Running Step 3: Final Plate Convergence...")
step3_features = filter_step3_plate_convergence(df_step2, consistent_features, top_n_to_keep=200)
df_step3 = df_step2[['Plate', 'Well_ID', 'Treatment'] + step3_features]

# ==========================================
# STEP 4: REDUNDANCY REMOVAL
# ==========================================
def filter_step4_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    # Mask the lower triangle of the correlation matrix
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

print("Running Step 4: Redundancy Filter...")
non_redundant_features = filter_step4_redundancy(df_step3, step3_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final_vetted = df_step3[['Plate', 'Well_ID', 'Treatment'] + non_redundant_features]
output_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles20_simpler.csv")
df_final_vetted.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Original features: {len(feature_cols)}")
print(f"Final feature count: {len(non_redundant_features)}")
print(f"Master profiles saved to: {output_path}")
print("="*40)

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_final_master_profiles20_simpler.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "UMAP_20simpler")

# Create the output directory if it doesn't exist
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x in ["no_sgRNA", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'UMAP1': False, 'UMAP2': False},
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) # Tiny mutants
    fig.update_traces(marker=dict(size=8, line=dict(width=1)), selector=dict(marker_symbol='square')) # Clear controls
    
    fig.update_layout(
        width=1000, 
        height=700,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Show the plot (In Jupyter, these will appear one after another)
    fig.show()
    
     # --- SAVE TO LOCATION ---
    save_path = os.path.join(OUTPUT_DIR, f"UMAP20simpler_{config['name']}.html")
    fig.write_html(save_path)
    
    fig.show()
    

In [ ]:
from scipy.linalg import inv, sqrtm

import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal/"
INPUT_CSV = os.path.join(PROJECT_ROOT, "aggregated_wells_data.csv")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)

# Ensure all columns are strings to avoid indexing issues
df_raw.columns = [str(c) for c in df_raw.columns]
feature_cols = [str(i) for i in range(6400)]

# ==========================================
# STEP 1: ACROSS-PLATE STABILITY
# ==========================================
def filter_step1_across_plate_stability(df, features, top_n_to_keep=800):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    # Basic variance check to remove dead/constant features
    variance_filter = df[features].std() > 0.01
    
    stable_features = total_batch_noise[variance_filter].sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

print("\nRunning Step 1: Across-Plate Stability...")
stable_cols = filter_step1_across_plate_stability(df_raw, feature_cols, top_n_to_keep=800)
df_step1 = df_raw[['Plate', 'Well_ID', 'Treatment'] + stable_cols]

# ==========================================
# STEP 2: WITHIN-PLATE CONSISTENCY (Former 4.5)
# ==========================================
def filter_within_plate_consistency(df, features, top_n_to_keep=500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    # Calculate standard deviation within each plate for controls
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

print("Running Step 2: Within-Plate Consistency...")
# FIX: Changed 'vetted_features_s4' to 'stable_cols'
consistent_features = filter_within_plate_consistency(df_step1, stable_cols, top_n_to_keep=500)
df_step2 = df_step1[['Plate', 'Well_ID', 'Treatment'] + consistent_features]

# ==========================================
# STEP 3: FINAL PLATE CONVERGENCE (Former 11)
# ==========================================
def filter_step3_plate_convergence(df, features, top_n_to_keep=200):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_noises.append(noise)
    
    avg_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)
    plate_blind_features = avg_plate_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return plate_blind_features

print("Running Step 3: Final Plate Convergence...")
step3_features = filter_step3_plate_convergence(df_step2, consistent_features, top_n_to_keep=200)
df_step3 = df_step2[['Plate', 'Well_ID', 'Treatment'] + step3_features]

# ==========================================
# STEP 4: REDUNDANCY REMOVAL
# ==========================================
def filter_step4_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    # Mask the lower triangle of the correlation matrix
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

print("Running Step 4: Redundancy Filter...")
non_redundant_features = filter_step4_redundancy(df_step3, step3_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final_vetted = df_step3[['Plate', 'Well_ID', 'Treatment'] + non_redundant_features]





# ==========================================
# STEP 5: SPHERING (WHITENING)
# ==========================================
def apply_sphering(df, features, control_label, reference_suffix='T0', epsilon=1e-6):
    """
    Applies ZCA whitening using the control_label at reference_suffix as the basis.
    """
    print(f"Applying Sphering based on {control_label} at {reference_suffix}...")
    
    # 1. Isolate the reference population
    ref_mask = (df['Treatment'] == control_label) & (df['Plate'].str.endswith(reference_suffix))
    ref_data = df.loc[ref_mask, features].values
    
    # 2. Center the entire dataset based on reference mean
    ref_mean = np.mean(ref_data, axis=0)
    X = df[features].values - ref_mean
    X_ref_centered = ref_data - ref_mean
    
    # 3. Calculate Covariance Matrix of reference
    # Adding epsilon to the diagonal (regularization) prevents singularity
    cov = np.cov(X_ref_centered, rowvar=False) + np.eye(len(features)) * epsilon
    
    # 4. Calculate Whitening Matrix (ZCA)
    # W = Q * Λ^(-1/2) * Q.T
    evals, evecs = np.linalg.eigh(cov)
    # Ensure eigenvalues are positive
    evals = np.maximum(evals, 1e-9) 
    whitening_matrix = evecs @ np.diag(1.0 / np.sqrt(evals)) @ evecs.T
    
    # 5. Transform the data
    X_sphered = X @ whitening_matrix
    
    # 6. Put back into dataframe
    df_sphered = df.copy()
    df_sphered[features] = X_sphered
    return df_sphered

# Apply the sphering to your non-redundant features
df_final_vetted = apply_sphering(
    df_step3, 
    non_redundant_features, 
    CONTROL_LABEL, 
    reference_suffix='T0'
)

# ==========================================
# FINAL SAVE (Updated)
# ==========================================
output_path = os.path.join(PROJECT_ROOT, "vetted_sphered_master_profiles.csv")
df_final_vetted.to_csv(output_path, index=False)

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = "/media/arnout/Elements1/groteEdeepprofilerdingen/DataDeepprofiler/finaal"
file_path = os.path.join(PROJECT_ROOT, "vetted_sphered_master_profiles.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "UMAP_20sphered")

# Create the output directory if it doesn't exist
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x in ["no_sgRNA", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']}...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'UMAP1': False, 'UMAP2': False},
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) # Tiny mutants
    fig.update_traces(marker=dict(size=8, line=dict(width=1)), selector=dict(marker_symbol='square')) # Clear controls
    
    fig.update_layout(
        width=1000, 
        height=700,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Show the plot (In Jupyter, these will appear one after another)
    fig.show()
    
     # --- SAVE TO LOCATION ---
    save_path = os.path.join(OUTPUT_DIR, f"UMAP20sphered_{config['name']}.html")
    fig.write_html(save_path)
    
    fig.show()
    